# Linear benchmarks (Elastic Net + OLS) — per-security, horizon-sweep, Train+OOS, block-CV robustness

**Purpose.** This notebook fits two *linear* benchmark estimators — ordinary least
squares (**OLS**) and the penalised **Elastic Net** — on **exactly the same data,
folds, features, cross-market PCA, target, purge/embargo and metric schema** as the
HTBoost per-security notebook (`model_htboost_v5_clean.ipynb`). Only the estimator
changes. Every results row is produced by the **shared** `htb_metrics` contract, so the
linear tables concatenate with `htboost_results_v5_nor/v5_metrics_long.csv` with zero
reconciliation and drop straight into Chapters 7–8 beside HTBoost.

The single design rule is **comparability**: the universe, the five walk-forward folds
and their regimes, the `_features_for_security_v4` feature set, the train-only
cross-market PCA, the `y = r_{t+h} − r_t` target, the `OVERLAP = h−1` purge, the
two-sided block-CV/LORO purge+embargo, the RNG seed and the `SHARED_COLS` schema are all
**reused verbatim** from v5. They are *not* redefined here.

**What is added.** A `model_kind` column distinguishes the rows (`OLS` / `ElasticNet`,
versus HTBoost's `per_security`). Elastic Net's `alpha` and `l1_ratio` are selected by
**internal time-series CV on the training rows of each fold only** (never on OOS/regime
folds); features are standardised inside the fit path (scaler fit on train, applied to
test) — required for a penalised model and kept leakage-free.

**Null-confirmation discipline.** Nothing is tuned on OOS or regime folds. In-sample fit
is reported as a diagnostic, never editorialised as forecasting skill. The heavy
multi-horizon sweeps are gated behind `RUN_FULL_SWEEP` (default **OFF**): by default this
notebook runs only the leakage audit, the smoke test (both estimators) and the manifest.

> **No Julia required.** Unlike the HTBoost notebook, the linear pipeline is pure
> Python/scikit-learn — it runs in a standard scientific-Python kernel.

## Setup

In [ ]:
# ── Imports — identical scientific stack to v5, MINUS juliacall, PLUS sklearn linear
# The linear benchmarks need no Julia runtime. Everything else (numpy/pandas, statsmodels
# HAC, scipy tests, StandardScaler, PCA, multipletests) is the SAME as the HTBoost v5
# notebook so the shared pipeline cells run unchanged.
import re
import os
import sys
import json
import hashlib
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest, binom, norm
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
# Linear estimators + time-series internal CV (the ONLY estimator-specific imports)
from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit

# Put the repo root (dir containing src/) on sys.path so the flat root-level module(s)
# (thesis_style, htb_metrics) import whether cwd is the repo root or notebooks/.
_ROOT = os.getcwd()
while _ROOT != os.path.dirname(_ROOT) and not os.path.isdir(os.path.join(_ROOT, 'src')):
    _ROOT = os.path.dirname(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)
from src.data.bloomberg import load_data  # was: data_loader shim (broke from notebooks/ cwd)
# Shared thesis figure style (Okabe-Ito palette, vector PDF, despined axes, regime colours).
from thesis_style import apply_thesis_style, OKABE_ITO, REGIME_COLORS, SERIES_COLOR
apply_thesis_style()
# Semantic model->colour mapping, sourced from the shared Okabe-Ito palette.
CB = {'OLS': OKABE_ITO[0], 'ElasticNet': OKABE_ITO[3], 'HTBoost': OKABE_ITO[2],
      'chance': '#999999', 'zero': '#999999'}

print('Imports OK (linear v5: sklearn LinearRegression/ElasticNetCV/TimeSeriesSplit; '
      'no juliacall)')

### Configuration
All run parameters in one place: horizons, the regularization grid, the walk-forward and block-CV fold dates, the PCA settings, minimum-observation thresholds, and the switches that gate the heavy sweeps. None of these are chosen from out-of-sample results.

In [ ]:
# ── CONFIG — all run parameters live here ────────────────────────────────────────────────
# Horizon grid, PCA compression, the pre-committed inner-CV grid, block-CV folds +
# embargo, and the run switches.
# NOTHING here is chosen by looking at an OOS/regime fold.

# ── Primary horizon + the pre-committed horizon GRID ───────────────────
H                = 5            # default/primary horizon (smoke, leakage audit, Part C default)
# Shared run-protocol constants now come from src.config (single source of truth);
# values are unchanged. H, OVERLAP, PCA/LAM, run switches etc. remain notebook-local.
from src.config import (H_GRID, MIN_TRAIN_OBS, MIN_TEST_OBS, ALPHA_MT,
                        BLOCK_CV_FOLDS, EMBARGO_FOR_H, WF_FOLDS, FOLD_NAMES)
H_GRID_LONG      = [126, 252]              # 6m, 12m — only if data length supports the folds
OVERLAP          = H - 1                    # label-overlap purge; ALWAYS H-1 per horizon (invariant)

LOSS             = 't'
MODALITY         = 'accurate'  # strongest HTBoost setting (committed scientific config)
UNIVERSE_MIN_OBS = 500
SMOKE_SEC        = 'NOR_10Y'   # NOR pilot (this study restricts to Norwegian swaps)
TARGET_LAGS      = 5           # AR lags of chg_1d; fixed, identical for all securities/horizons

OUT_DIR = 'htboost_results_v5_nor'
os.makedirs(OUT_DIR, exist_ok=True)

# ── PCA compression of the cross-market xm_* block (fit on TRAIN only) ───
# The xm_* block (~178 cols for NOR_10Y) is the named overfitting surface. We
# replace it with k principal components, fit on the fold's TRAINING rows only,
# applied identically to every tenor/horizon/fold. Pre-committed rule:
#   k = smallest #components explaining ≥ XM_PCA_VAR of TRAIN variance, capped at XM_PCA_KMAX.
# Standardisation uses TRAIN mean/std. This lives in the FIT PATH, never in
# _features_for_security_v4 (the audit cannot validate a fold-dependent transform).
XM_PCA_ENABLE    = True
XM_PCA_VAR       = 0.95        # explained-variance target (a-priori)
XM_PCA_KMAX      = 12          # hard cap on #components (a-priori)

# ── L2 regularization fixed A PRIORI (no data-driven lambda tuning) ──
# For a null-confirmation harness we do NOT tune lambda. It is pre-committed to the
# MIDPOINT of LAM_GRID and applied uniformly to every horizon/tenor/fold/scheme;
# robustness across lambda is REPORTED by the λ-sensitivity cell (never a selection).
# depth / ntrees are selected by modality='accurate's internal CV per fit (a-priori
# uniform rule); gridding depth would double-CV it (documented HTBoost behaviour).
LAM_GRID         = [0.05, 0.10, 0.20]   # lambda grid: midpoint pre-committed; rest for the sensitivity report
NFOLD_INTERNAL   = 4                    # raised above 1 → genuine internal block-CV early stopping
# FROZEN_CONFIG is set in the regularization cell (lambda = LAM_GRID midpoint) and reused EVERYWHERE.
LAM              = 0.10                  # provisional; set to the a-priori midpoint in the regularization cell
FROZEN_CONFIG    = None                 # set by the regularization cell; dict(lam, depth, nfold, modality, loss)

# ── block-CV / leave-one-regime-out folds + embargo ─────────────────────
# Full sample (2007+), aligned to regimes. EMBARGO scales with H.
# BLOCK_CV_FOLDS + EMBARGO_FOR_H imported from src.config (see top of cell).

# ── Run switches (gate the heavy sweeps) ─────────────────────────────────────
RUN_WALK_FORWARD = True
RUN_BLOCK_CV     = True
RUN_LORO         = True
# QUICK_MODE is a DEVELOPMENT/EXECUTION-BUDGET switch only — it does NOT change any
# model-selection decision (those stay frozen). It shrinks modality / grid / subset
# so the whole notebook runs end-to-end quickly to verify wiring. Default OFF: the
# thesis run uses the committed config. When ON, results are NOT for reporting.
QUICK_MODE       = False
QUICK_MODALITY   = 'fast'
QUICK_TENORS     = ['NOR_10Y']
QUICK_HGRID      = [5]

# ── pooled-notebook comparison ──────────────────────────────────────────
POOLED_CSV_PATH  = f'{OUT_DIR}/pooled_metrics_long.csv'   # configurable; absent ⇒ comparison skips

# ── Norway external-data fetch (Part B) ───────────────────────────────────────
NOR_FETCH        = os.environ.get('NORWAY_LIVE_FETCH', '') == '1'   # cache-first default; set NORWAY_LIVE_FETCH=1 to refresh from live APIs
NOR_CACHE        = f'{_ROOT}/data/cache/norway_raw_features.csv'     # git-shipped cache, cwd-independent; a live refresh overwrites it
NOR_RELEASE_LAG_M = 1

# ── Walk-forward folds: expanding window, test windows map to regimes ─────────
# WF_FOLDS + FOLD_NAMES imported from src.config (see top of cell).
NOTEBOOK_TAG = 'v5'
RUN_TS = pd.Timestamp.now(tz='UTC').isoformat()

# Seed the cache from the existing v4 cache so the Norway fetch can run offline
# rather than dropping NOR features when the live sources are unreachable.
if not os.path.exists(NOR_CACHE) and os.path.exists('htboost_results_v4_nor/norway_raw_features.csv'):
    import shutil; shutil.copy('htboost_results_v4_nor/norway_raw_features.csv', NOR_CACHE)

print(f'H(primary)={H}  H_GRID={H_GRID}  (+long {H_GRID_LONG} if data supports)')
print(f'LOSS={LOSS!r} MODALITY={MODALITY!r}  NFOLD_INTERNAL={NFOLD_INTERNAL}  TARGET_LAGS={TARGET_LAGS}')
print(f'PCA(xm_*): enable={XM_PCA_ENABLE} var≥{XM_PCA_VAR} kmax={XM_PCA_KMAX}')
print(f'inner-CV LAM grid: {LAM_GRID}  (depth/ntrees via modality internal CV)')
print(f'switches: WF={RUN_WALK_FORWARD} BLOCK_CV={RUN_BLOCK_CV} LORO={RUN_LORO}  QUICK_MODE={QUICK_MODE}')
print(f'OUT_DIR={OUT_DIR!r}')
print(f'Walk-forward folds ({len(WF_FOLDS)}):')
for f in WF_FOLDS:
    print(f'  {f[0]:<14s}  test [{f[1]} → {f[2]}]  regime={f[3]}')
print(f'Block-CV folds ({len(BLOCK_CV_FOLDS)}):')
for f in BLOCK_CV_FOLDS:
    print(f'  {f[0]:<14s}  block [{f[1]} → {f[2]}]  regime={f[3]}')

In [ ]:
# ── LINEAR override of the shared CONFIG ─────────────────────────────────────────────
# The cell ABOVE is v5's CONFIG reused VERBATIM, so H_GRID / H_GRID_LONG / OVERLAP,
# WF_FOLDS, BLOCK_CV_FOLDS, EMBARGO_FOR_H, the PCA rule (XM_PCA_*), TARGET_LAGS,
# MIN_TRAIN_OBS / MIN_TEST_OBS, NFOLD_INTERNAL and ALPHA_MT are IDENTICAL to HTBoost.
# Here we change ONLY what is estimator- or output-specific. Nothing below touches a
# fold definition, the target, the PCA rule, or the metric schema.

# Output location (mirrors htboost_results_v5_nor/ with a lin_ prefix on every file).
OUT_DIR = 'linear_results_v5_nor'
FIG_DIR = f'{OUT_DIR}/figures'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Provenance tag for the shared schema (HTBoost uses 'v5').
NOTEBOOK_TAG = 'linear_v5'
RUN_TS = pd.Timestamp.now(tz='UTC').isoformat()

# The two estimators. model_kind enters the SHARED schema's `model_kind` column and is
# the row-distinguishing key (HTBoost rows carry 'per_security').
MODEL_KINDS = ['OLS', 'ElasticNet']

# Deterministic RNG seed (matches the spirit of v5's pinned Julia seed 20260619).
SEED = 20260619
np.random.seed(SEED)

# ── Elastic Net internal-CV grid (selected on TRAIN rows of each fold ONLY) ──────────
# l1_ratio grid per the agent brief; alphas auto-selected by ElasticNetCV along its path.
# cv uses TimeSeriesSplit (block/time-series, NOT random shuffling) with NFOLD_INTERNAL
# splits — consistent with v5's internal-CV style. This is the ONLY tuning in the whole
# notebook and it never sees an OOS/regime fold.
EN_L1_GRID   = [0.1, 0.5, 0.7, 0.9, 0.95, 1.0]
EN_N_ALPHAS  = 50                      # alphas along each l1_ratio path (a-priori)
EN_MAX_ITER  = 5000                    # coordinate-descent cap; collinear folds may warn
EN_TOL       = 1e-3                    # convergence tol (looser → tractable on collinear X)
EN_CV_SPLITS = NFOLD_INTERNAL          # inherited from v5 CONFIG (=4)

# Per-model frozen config dicts → distinct config_hash per estimator (no data-driven
# selection for OLS; ElasticNet's penalty is selected internally on train only).
def lin_cfg(model_kind):
    if model_kind == 'OLS':
        return {'model': 'OLS', 'standardize': True}
    if model_kind == 'ElasticNet':
        return {'model': 'ElasticNet', 'standardize': True,
                'l1_ratio_grid': EN_L1_GRID, 'n_alphas': EN_N_ALPHAS,
                'cv': f'TimeSeriesSplit({EN_CV_SPLITS})', 'max_iter': EN_MAX_ITER}
    raise ValueError(model_kind)

# ── Run switches ─────────────────────────────────────────────────────────────────────
# Default OFF: the notebook runs ONLY the leakage audit + smoke (both estimators) +
# manifest. Flip RUN_FULL_SWEEP=True to launch the heavy multi-horizon sweeps (linear
# fits are fast — seconds, not minutes — but we still gate them per the study's
# null-confirmation discipline and the agent brief's "do not launch unless asked").
RUN_FULL_SWEEP   = False
RUN_WALK_FORWARD = RUN_FULL_SWEEP
RUN_BLOCK_CV     = RUN_FULL_SWEEP
RUN_LORO         = RUN_FULL_SWEEP

# Point the Norway cache at the linear OUT_DIR and seed it from the v4 cache so Part B
# runs offline (identical raw NOR features to v5).
NOR_CACHE = f'{_ROOT}/data/cache/norway_raw_features.csv'
if not os.path.exists(NOR_CACHE):
    for _src in (f'htboost_results_v5_nor/norway_raw_features.csv',
                 'htboost_results_v4_nor/norway_raw_features.csv'):
        if os.path.exists(_src):
            import shutil; shutil.copy(_src, NOR_CACHE); break

# Path to the HTBoost results (for the optional cross-model overlay figure).
HTB_METRICS_CSV = 'htboost_results_v5_nor/v5_metrics_long.csv'

print('=== linear override applied ===')
print(f'  OUT_DIR={OUT_DIR!r}  FIG_DIR={FIG_DIR!r}')
print(f'  MODEL_KINDS={MODEL_KINDS}  SEED={SEED}')
print(f'  ElasticNet l1_ratio grid={EN_L1_GRID}  n_alphas={EN_N_ALPHAS}  '
      f'cv=TimeSeriesSplit({EN_CV_SPLITS})')
print(f'  RUN_FULL_SWEEP={RUN_FULL_SWEEP}  (WF={RUN_WALK_FORWARD} '
      f'BLOCK_CV={RUN_BLOCK_CV} LORO={RUN_LORO})')
print(f'  Folds / PCA / horizons / target / purge inherited VERBATIM from v5 CONFIG.')
print(f'  HTBoost overlay source: {HTB_METRICS_CSV} '
      f'({"present" if os.path.exists(HTB_METRICS_CSV) else "ABSENT — overlay fig will be skipped"})')

### Load data and define the universe
Load the swap panel, identify the swap columns, guard against duration columns leaking in, and restrict the study universe to Norwegian swaps with enough history.

In [ ]:
df_raw = load_data()

_SWAP_PAT = re.compile(r'^[A-Z]+_\d+[WMY]$')
SWAP_COLS  = sorted([c for c in df_raw.columns if _SWAP_PAT.match(c)])

# Duration.xlsx is NOT loaded by load_data() — guard anyway
dur_cols = [c for c in df_raw.columns if 'uration' in c.lower() or 'DV01' in c]
assert len(dur_cols) == 0, f'Duration columns found — exclude: {dur_cols}'

print(f'df_raw shape:    {df_raw.shape}')
print(f'Date range:      {df_raw.index.min().date()} → {df_raw.index.max().date()}')
print(f'Swap columns:    {len(SWAP_COLS)}')
print(f'Non-swap columns:{len(df_raw.columns) - len(SWAP_COLS)}')
print(f'Duration guard:  OK (0 duration columns in df_raw)')

# All available (non-swap) column names — inventory of what we can use as features
non_swap = [c for c in df_raw.columns if not _SWAP_PAT.match(c)]
print(f'\nNon-swap columns available for features ({len(non_swap)}):')
for c in non_swap:
    n = df_raw[c].notna().sum()
    print(f'  {c:<40s}  {n:5d} obs')

# Universe: freeze on full history (all available obs)
obs      = df_raw[SWAP_COLS].notna().sum()
UNIVERSE = sorted(obs[obs >= UNIVERSE_MIN_OBS].index.tolist())
print(f'\nUniverse (≥{UNIVERSE_MIN_OBS} obs): {len(UNIVERSE)} securities')


# ── PART A: restrict universe to Norwegian swaps ─────────────────────────────
UNIVERSE = [s for s in UNIVERSE if s.rsplit('_', 1)[0] == 'NOR']
print(f'\nPART A — Norwegian-swap universe ({len(UNIVERSE)} securities):')
for s in UNIVERSE:
    print(f'  {s:<10s}  {df_raw[s].notna().sum():5d} obs')
assert SMOKE_SEC in UNIVERSE, f'{SMOKE_SEC} not in NOR universe'

### Part B — Norway data fetch
Probe each public Norwegian/European data source live (Norges Bank, SSB, ECB, Riksbank), log what connects, and derive leakage-safe `nor_*` feature columns. Sources that fail are skipped, with the existing columns as the fallback.

In [ ]:
# ── PART B: Norway-specific data fetch + connectivity report ─────────────────
# Probe every source live. Integrate a series ONLY if the fetch returns usable data.
# Anything that fails is LOGGED and skipped (existing columns are the fallback).
#   Norges Bank (no auth):  EXR (EUR/NOK, USD/NOK, I-44), IR (policy rate),
#                           SHORT_RATES (NOWA), GOVT_GENERIC_RATES (govt yields)
#   SSB PxWebApi (no auth):  KPI YoY (03013), KPI-JAE YoY (05327), LFS unemp (13760)
#   ECB SDMX:                deposit facility rate;   Riksbank SWEA: policy rate
#   Brent / Brent-WTI / EIA gas:  SKIPPED (needs an API key) -> fall back to the
#                           existing oil_mom_* / natgas_mom_1m columns (already shift-1).
import io, urllib.request

NB_API  = 'https://data.norges-bank.no/api/data'
SSB_API = 'https://data.ssb.no/api/v0/en/table'
_HDRS   = {'User-Agent': 'master-thesis-research'}


def _http_get(url):
    return urllib.request.urlopen(urllib.request.Request(url, headers=_HDRS),
                                  timeout=30).read().decode('utf-8', 'replace')


def _http_post(url, payload):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={**_HDRS, 'Content-Type': 'application/json'})
    return urllib.request.urlopen(req, timeout=30).read().decode('utf-8', 'replace')


def _nb_series(flow, key, start, end):
    """Norges Bank SDMX -> pd.Series(OBS_VALUE) indexed by date."""
    url = f'{NB_API}/{flow}/{key}?format=csv&startPeriod={start}&endPeriod={end}'
    d = pd.read_csv(io.StringIO(_http_get(url)), sep=';')
    s = pd.Series(pd.to_numeric(d['OBS_VALUE'], errors='coerce').values,
                  index=pd.to_datetime(d['TIME_PERIOD'])).dropna().sort_index()
    return s[~s.index.duplicated(keep='last')]


def _ssb_monthly(table, content, filters, start, end):
    """SSB PxWebApi monthly -> pd.Series indexed by reference-month END timestamp."""
    query = [{'code': c, 'selection': {'filter': 'item', 'values': v}} for c, v in filters]
    query.append({'code': 'ContentsCode', 'selection': {'filter': 'item', 'values': [content]}})
    js  = json.loads(_http_post(f'{SSB_API}/{table}',
                                {'query': query, 'response': {'format': 'json-stat2'}}))
    idx, vals = js['dimension']['Tid']['category']['index'], js['value']
    out = {}
    for m, pos in sorted(idx.items(), key=lambda kv: kv[1]):
        if vals[pos] is None:
            continue
        out[pd.Timestamp(int(m[:4]), int(m[5:7]), 1) + pd.offsets.MonthEnd(0)] = float(vals[pos])
    s = pd.Series(out).sort_index()
    return s[(s.index >= start) & (s.index <= end)]


def _ecb_dfr(start):
    url = ('https://data-api.ecb.europa.eu/service/data/FM/'
           f'B.U2.EUR.4F.KR.DFR.LEV?format=csvdata&startPeriod={start}')
    d = pd.read_csv(io.StringIO(_http_get(url)))
    return pd.Series(pd.to_numeric(d['OBS_VALUE'], errors='coerce').values,
                     index=pd.to_datetime(d['TIME_PERIOD'])).dropna().sort_index()


def _riksbank(start):
    j = json.loads(_http_get(f'https://api.riksbank.se/swea/v1/Observations/SECBREPOEFF/{start}'))
    return pd.Series({pd.Timestamp(o['date']): float(o['value']) for o in j}).sort_index()


def fetch_norway_data(start, end):
    """Probe every Norway source. Returns (raw_df, report). Each source is wrapped:
    on failure it is logged and skipped, never fabricated."""
    a, b   = pd.Timestamp(start).strftime('%Y-%m-%d'), pd.Timestamp(end).strftime('%Y-%m-%d')
    raw, report = {}, []

    def _try(label, fn, freq):
        try:
            s = fn()
            d = s.dropna() if s is not None else pd.Series(dtype=float)
            if len(d) == 0:
                report.append((label, 'SKIP', 'empty', freq, 0, None, None)); return None
            report.append((label, 'OK', '', freq, len(d), d.index.min().date(), d.index.max().date()))
            return s
        except Exception as e:
            report.append((label, 'FAIL', repr(e)[:60], freq, 0, None, None)); return None

    raw['nb_eurnok'] = _try('NB EUR/NOK',  lambda: _nb_series('EXR', 'B.EUR.NOK.SP', a, b), 'daily')
    raw['nb_usdnok'] = _try('NB USD/NOK',  lambda: _nb_series('EXR', 'B.USD.NOK.SP', a, b), 'daily')
    raw['nb_i44']    = _try('NB I-44',     lambda: _nb_series('EXR', 'B.I44.NOK.SP', a, b), 'daily')
    raw['nb_polrate']= _try('NB policy rate', lambda: _nb_series('IR', 'B.KPRA.SD.R', a, b), 'daily')
    raw['nb_nowa']   = _try('NB NOWA',     lambda: _nb_series('SHORT_RATES', 'B.NOWA.ON.R', a, b), 'daily')
    for ten in ['3Y', '5Y', '10Y']:
        raw[f'nb_govt_{ten.lower()}'] = _try(
            f'NB govt {ten}', lambda t=ten: _nb_series('GOVT_GENERIC_RATES', f'B.{t}.GBON', a, b), 'daily')
    raw['ssb_kpi_yoy']   = _try('SSB KPI YoY', lambda: _ssb_monthly(
        '03013', 'Tolvmanedersendring', [('Konsumgrp', ['TOTAL'])], a, b), 'monthly')
    raw['ssb_kpijae_yoy']= _try('SSB KPI-JAE YoY', lambda: _ssb_monthly(
        '05327', 'Tolvmanedersendring', [('Konsumgrp', ['JAE_TOTAL'])], a, b), 'monthly')
    raw['ssb_unemp']     = _try('SSB LFS unemp', lambda: _ssb_monthly(
        '13760', 'Arbeidsledige',
        [('Kjonn', ['0']), ('Alder', ['15-74']), ('Justering', ['S'])], a, b), 'monthly')
    raw['ecb_dfr']    = _try('ECB deposit rate', lambda: _ecb_dfr(a), 'event')
    raw['rb_polrate'] = _try('Riksbank policy rate', lambda: _riksbank(a), 'daily')

    raw = {k: v for k, v in raw.items() if v is not None}
    return (pd.DataFrame(raw).sort_index() if raw else pd.DataFrame()), report


def _monthly_released_daily(s_refmonth, target_index, lag_m=NOR_RELEASE_LAG_M):
    """Reference-month value -> stamped at end of month M+lag (>= real SSB release),
    then ffill onto daily target_index. Conservative: never look-ahead."""
    if s_refmonth is None or len(s_refmonth.dropna()) == 0:
        return pd.Series(np.nan, index=target_index)
    sh = s_refmonth.dropna().sort_index().copy()
    sh.index = sh.index + pd.offsets.MonthEnd(lag_m)
    sh = sh[~sh.index.duplicated(keep='last')]
    return sh.reindex(target_index, method='ffill')


def add_norway_features_v4(df, nor_raw=None):
    """Leakage-safe NOR-gated feature columns (nor_*) from raw fetched series.
    Timing policy:
      FX (EUR/NOK, USD/NOK, I-44) ........ shift(1)  [USD-cross & index, prior close]
      policy rate / NOWA / govt yields ... contemporaneous  [NOK official same-session]
      ECB / Riksbank policy rates ........ contemporaneous  [administered step, ffill]
      monthly macro (KPI, KPI-JAE, unemp) release-aligned ffill + shift(1)
    Daily series read from df (joined); monthly series from nor_raw (native month index)."""
    df  = df.copy()
    idx = df.index

    def _raw(col):
        return df[col].copy() if col in df.columns else pd.Series(np.nan, index=idx)

    # FX — shift(1)
    for srccol, lbl in [('nb_eurnok', 'eurnok'), ('nb_usdnok', 'usdnok'), ('nb_i44', 'i44')]:
        fx = _raw(srccol)
        if fx.notna().sum() == 0:
            continue
        df[f'nor_{lbl}_level']    = fx.shift(1)
        df[f'nor_{lbl}_chg_1d']   = fx.diff().shift(1)
        df[f'nor_{lbl}_mom_1m']   = fx.pct_change().rolling(21, min_periods=10).sum().shift(1)
        df[f'nor_{lbl}_mom_3m']   = fx.pct_change().rolling(63, min_periods=30).sum().shift(1)
        df[f'nor_{lbl}_rvol_20d'] = fx.pct_change().rolling(20, min_periods=5).std().shift(1)

    # Policy rate (styringsrenten) — contemporaneous; days-since-last-change
    pol = _raw('nb_polrate')
    if pol.notna().sum() > 0:
        df['nor_polrate_level']  = pol
        df['nor_polrate_chg_1d'] = pol.diff()
        grp = (pol.diff().fillna(0) != 0).cumsum()
        df['nor_polrate_days_since_chg'] = grp.groupby(grp).cumcount().astype(float)

    # NOWA + NIBOR-NOWA spread — contemporaneous (NOK same-session)
    nowa = _raw('nb_nowa')
    if nowa.notna().sum() > 0:
        df['nor_nowa_level']        = nowa
        df['nor_nibor_nowa_spread'] = _raw('NIBOR3M Index  (L1)') - nowa

    # Government bond yields — contemporaneous (NOK official)
    for ten in ['3y', '5y', '10y']:
        g = _raw(f'nb_govt_{ten}')
        if g.notna().sum() > 0:
            df[f'nor_govt_{ten}'] = g

    # ECB deposit rate / Riksbank policy rate — contemporaneous (administered step)
    ecb = _raw('ecb_dfr').reindex(idx).ffill()
    if ecb.notna().sum() > 0:
        df['nor_ecb_dfr_level'] = ecb
        df['nor_ecb_dfr_chg']   = ecb.diff()
    rb = _raw('rb_polrate').reindex(idx).ffill()
    if rb.notna().sum() > 0:
        df['nor_rb_rate_level'] = rb
        df['nor_rb_rate_chg']   = rb.diff()

    # Monthly macro — release-aligned ffill + shift(1)
    for srccol, dst in [('ssb_kpi_yoy', 'nor_kpi_yoy'),
                        ('ssb_kpijae_yoy', 'nor_kpijae_yoy'),
                        ('ssb_unemp', 'nor_unemp')]:
        mser = nor_raw[srccol] if (nor_raw is not None and srccol in nor_raw.columns) else None
        if mser is not None and mser.dropna().shape[0] > 0:
            df[dst] = _monthly_released_daily(mser, idx).shift(1)
    return df


# ── Execute fetch (live, with cache fallback) ────────────────────────────────
nor_raw = pd.DataFrame()
nor_report = []
if NOR_FETCH:
    try:
        nor_raw, nor_report = fetch_norway_data(df_raw.index.min(), df_raw.index.max())
        if not nor_raw.empty:
            nor_raw.to_csv(NOR_CACHE)
    except Exception as e:
        print(f'[WARN] live fetch failed ({repr(e)[:80]}); trying cache')
if nor_raw.empty and os.path.exists(NOR_CACHE):
    nor_raw = pd.read_csv(NOR_CACHE, index_col=0, parse_dates=True)
    nor_report = nor_report or [('cache load', 'OK', NOR_CACHE, 'mixed', len(nor_raw), None, None)]

print('=== PART B — connectivity report ===')
print(f'  {"source":24s} {"stat":4s} {"freq":7s} {"n_obs":>6s}  coverage')
print('  ' + '-' * 70)
for label, status, note, freq, n, lo, hi in nor_report:
    cov = f'{lo}..{hi}' if lo else ''
    print(f'  {label:24s} {status:4s} {freq:7s} {n:6d}  {cov}  {note}')
print('  SKIP Brent / Brent-WTI / EIA gas  (needs API key) -> fall back to oil_mom_*/natgas_mom_1m')

# Merge daily raw series into df_raw (monthly SSB series stay in nor_raw), then derive nor_*
_daily_raw = [c for c in nor_raw.columns if not c.startswith('ssb_')]
# Idempotent: only join raw daily series not already present (re-joining existing
# columns would raise an overlap error), then (re)derive nor_* from source columns.
_join_cols = [c for c in _daily_raw if c not in df_raw.columns]
if _join_cols:
    df_raw = df_raw.join(nor_raw[_join_cols], how='left')
df_raw = add_norway_features_v4(df_raw, nor_raw)
NOR_FEATURE_COLS = sorted(c for c in df_raw.columns if c.startswith('nor_'))
print(f'\n{len(NOR_FEATURE_COLS)} nor_* feature columns derived (gated to country==\'NOR\'):')
for c in NOR_FEATURE_COLS:
    print(f'  {c:30s} {df_raw[c].notna().sum():6d} obs')

### Macro/global feature builder
Define the timing-alignment policy and `add_macro_features_v4`, which builds the global macro, volatility, equity, commodity, credit and IBOR features. The policy lag keeps every exogenous feature observable at decision time.

In [ ]:
# ── TIMING ALIGNMENT POLICY ──────────────────────────────────────
# US-close exogenous series (VIX/MOVE/equity/commodity) are shifted by 1 day so a
# feature is always observable at decision time. Same-day US-close values are safe
# for US securities (SOFR) where feature and fixing are both US-close, but for
# non-US fixings (EUR, NOR fix intraday before NY close) the same-day US close is
# not yet available. Shifting US-close series by 1 day — feature[t] = series[t-1] —
# makes every feature observable at any market.
#
# Series NOT shifted (contemporaneous OK):
#   - own-security swap rate: target and feature are the same daily close
#   - country IBOR rates: same-country same-session fix
#   - monthly macro: ffill+shift(1)
#
# Series shifted by 1:
#   - VIX, MOVE, V2X, VXN, RVX, OVX, GVZ
#   - SPX, SX5E, MXWO, NDX
#   - Oil, copper, OPEC, natgas
#   - Breakeven inflation (TIPS), IG/HY credit spreads
#   - Additional: CPURNSA, PCE CORE
#
# Cross-market swap rates: shift(1) applied in _features_for_security_v4.

MATURITY_YEARS = {
    '1W': 1/52, '1M': 1/12, '3M': 0.25, '6M': 0.5,
    '1Y': 1.0,  '5Y': 5.0,  '10Y': 10.0, '15Y': 15.0, '30Y': 30.0,
}
COUNTRY_PRIMARY_IBOR = {
    'NOR':   ('NIBOR3M Index  (L1)',  'NIBOR6M Index'),
    'SWE':   ('STIB3M Index  (L1)',   None),
    'EUR':   ('EUR006M Index  (L1)',  'EUR003M Index'),
    'SOFR':  ('SOFRRATE Index  (L1)', None),
    'CAN':   ('CAONREPO Index  (L1)', None),
    'AUS':   ('BBSW3M Index  (L1)',   None),
    'POL':   ('WIBR3M Index  (L1)',   None),
    'BRAZ':  ('BZDIOVRA Index  (R2)', None),
    'CHIN':  ('CNRR007 Index  (R2)',  None),
    'TURK':  ('MUTKCALM Index  (R1)', None),
    'SONIA': ('SONIO/N Index  (L1)',  None),
}


def add_macro_features_v4(df):
    df = df.copy()

    def _safe(col):
        return df[col].copy() if col in df.columns else pd.Series(np.nan, index=df.index)

    def _s1(col):
        return _safe(col).shift(1)

    def _mom_s1(col, w, mp):
        return _safe(col).pct_change().rolling(w, min_periods=mp).sum().shift(1)

    def _diff_roll_s1(col, w, mp):
        return _safe(col).diff(1).rolling(w, min_periods=mp).sum().shift(1)

    # MOVE — shift(1): US rates vol, only final after US close
    move = _safe('MOVE Index  (R3)')
    df['move_level']  = move.shift(1)
    df['move_chg_1d'] = move.diff().shift(1)
    mv_mu = move.rolling(252, min_periods=60).mean()
    mv_sd = move.rolling(252, min_periods=60).std()
    df['move_zscore'] = ((move - mv_mu) / (mv_sd + 1e-9)).clip(-5, 5).shift(1)

    # VIX — shift(1): US 4pm close
    vix = _safe('^VIX')
    df['vix_level']  = vix.shift(1)
    vx_mu = vix.rolling(252, min_periods=60).mean()
    vx_sd = vix.rolling(252, min_periods=60).std()
    df['vix_zscore'] = ((vix - vx_mu) / (vx_sd + 1e-9)).clip(-5, 5).shift(1)

    # Additional vol indices — shift(1)
    for src, dst in [('VXN Index  (R2)', 'vxn_level'),
                     ('RVX Index  (L2)', 'rvx_level'),
                     ('OVX Index  (R3)', 'ovx_level'),
                     ('GVZ Index  (R2)', 'gvz_level'),
                     ('V2X Index  (L2)', 'v2x_level')]:
        df[dst] = _s1(src)

    # Equity — shift(1)
    for src, lbl in [('SPX Index  (R4)',  'spx'),
                     ('SX5E Index  (R4)', 'sx5e'),
                     ('MXWO Index  (R4)', 'mxwo'),
                     ('NDX Index  (L4)',  'ndx')]:
        df[f'{lbl}_mom_1m'] = _mom_s1(src, 21, 10)
        df[f'{lbl}_mom_3m'] = _mom_s1(src, 63, 30)

    # Commodities — shift(1)
    df['oil_mom_1m']    = _mom_s1('CL1 COMB Comdty  (R3)', 21, 10)
    df['oil_mom_3m']    = _mom_s1('CL1 COMB Comdty  (R3)', 63, 30)
    df['copper_mom_1m'] = _mom_s1('C01 Comdty  (R4)', 21, 10)
    df['opec_prod']     = _s1('OPECDALY Index  (R3)')
    df['natgas_mom_1m'] = _mom_s1('MUC1 Comdty  (L2)', 21, 10)

    # Breakeven inflation — shift(1): TIPS market (US)
    for bc in ['breakeven_5Y', 'breakeven_10Y']:
        be = _safe(bc)
        df[f'{bc}_level']  = be.shift(1)
        df[f'{bc}_chg_1m'] = be.diff(21).shift(1)
    df['breakeven_slope'] = (_safe('breakeven_10Y') - _safe('breakeven_5Y')).shift(1)

    # Credit spreads — shift(1): US credit market
    df['ig_spread'] = _s1('IG_spread')
    df['hy_spread'] = _s1('HY_spread')
    df['ig_chg_1m'] = _safe('IG_spread').diff(21).shift(1)
    df['hy_chg_1m'] = _safe('HY_spread').diff(21).shift(1)

    # Extra inflation — shift(1)
    df['cpi_nsa_level']  = _s1('CPURNSA Index  (L3)')
    df['pce_core_level'] = _s1('PCE CORE Index  (L2)')

    # Monthly macro — ffill+shift(1): same logic as v3
    for src, dst in [
        ('CPI YOY Index  (R1)',  'cpi_yoy'),
        ('PCE CRCH Index  (L1)', 'pce_core_chg'),
        ('NFP TCH Index  (L4)',  'nfp_change'),
        ('NAPMPMI Index  (R2)',  'pmi_us'),
        ('ECCPEMUY Index  (R1)', 'pmi_eur'),
        ('UMRTEMU Index  (R1)',  'unemp_eur'),
    ]:
        df[dst] = _safe(src).ffill().shift(1)

    # Country IBOR rates — contemporaneous (own-country daily fix)
    for country, (col3m, col6m) in COUNTRY_PRIMARY_IBOR.items():
        sr3m = _safe(col3m)
        df[f'{country}_ibor_level']      = sr3m
        df[f'{country}_ibor_chg_1d']     = sr3m.diff()
        df[f'{country}_ibor_mom_1m']     = sr3m.diff().rolling(21, min_periods=10).sum()
        sr6m = _safe(col6m) if col6m else pd.Series(np.nan, index=df.index)
        df[f'{country}_ibor_term_slope'] = sr6m - sr3m

    # Additional EURIBOR + NIBOR tenors — contemporaneous
    for src, dst in [
        ('EUR003M Index', 'eur_3m'),
        ('EUR001W Index', 'eur_1w'),
        ('NIBOR1M Index', 'nor_ibor_1m'),
        ('NIBOR6M Index', 'nor_ibor_6m'),
    ]:
        df[dst] = _safe(src)

    return df


print('add_macro_features_v4 defined')

### Curve helpers
Small helpers for the swap annuity, linear curve interpolation, and the carry/roll spread used downstream.

In [ ]:
def swap_annuity(S_pct, T_years):
    if pd.isna(S_pct) or S_pct <= 0.01:
        return np.nan
    S_semi = S_pct / 200
    n = 2 * T_years
    return (1 - (1 + S_semi) ** (-n)) / (S_pct / 100)


def interp_rate(country, T_target, df):
    pts = {}
    for m, T in MATURITY_YEARS.items():
        col = f'{country}_{m}'
        if col in df.columns:
            pts[T] = df[col]
    if not pts:
        return pd.Series(np.nan, index=df.index)
    ts = sorted(pts.keys())
    if T_target <= 0 or T_target <= ts[0]:
        return pd.Series(0.0, index=df.index)
    if T_target >= ts[-1]:
        return pts[ts[-1]]
    for i in range(len(ts) - 1):
        if ts[i] <= T_target <= ts[i + 1]:
            w = (T_target - ts[i]) / (ts[i + 1] - ts[i])
            return pts[ts[i]] * (1 - w) + pts[ts[i + 1]] * w
    return pd.Series(np.nan, index=df.index)


def carry_roll_spread(sec, df, country, maturity):
    T = MATURITY_YEARS.get(maturity, np.nan)
    if np.isnan(T):
        return pd.Series(np.nan, index=df.index)
    S_n   = df[sec]
    T_roll = max(0.0, T - 1.0)
    S_nm1  = (pd.Series(0.0, index=df.index) if T_roll == 0.0
              else interp_rate(country, T_roll, df))
    return S_n - S_nm1


print('Helper functions defined: swap_annuity, interp_rate, carry_roll_spread')

### Per-security feature set
Assemble the full feature matrix for one security: own-curve momentum/vol/level features, contemporaneous IBOR, the lagged global block, and the cross-market swap-rate features.

In [ ]:
def _features_for_security_v4(sec, df, country, maturity):
    """Full feature set for one security.

    Reads precomputed columns from df (add_macro_features_v4 applied, so exogenous
    series already carry shift(1)), and adds:
      - AR target lags chg_lag1..chg_lagTARGET_LAGS: backward-looking by
        construction (lag-k at t uses chg_1d[t-k]); picked up by leakage audit.
      - Extra vol indices: V2X, OVX, VXN, RVX, GVZ (shift(1) via precomputed)
      - Credit spreads: IG, HY (shift(1) via precomputed)
      - Energy: OPEC prod, natgas (shift(1) via precomputed)
      - Extra IBOR: EUR 1w/3m, NOR 1m/6m (contemporaneous via precomputed)
      - Cross-market swap rates: ALL other swap columns, shift(1) applied here.
        feature[t] = other_swap[t-1] — safe for all market timezones.
    """

    def _get(col):
        return df[col].copy() if col in df.columns else pd.Series(np.nan, index=df.index)

    # ── Own-security features (contemporaneous) ───────────────────────────────
    level    = df[sec]
    chg_1d   = level.diff(1)

    # Explicit autoregressive target lags (lag-1..TARGET_LAGS of chg_1d).
    # Backward-looking by construction: lag-k at t uses chg_1d at t-k only.
    _tlags = {f'chg_lag{k}': chg_1d.shift(k) for k in range(1, TARGET_LAGS + 1)}

    mom_1m   = chg_1d.rolling(21,  min_periods=10).sum()
    mom_3m   = chg_1d.rolling(63,  min_periods=30).sum()
    mom_6m   = chg_1d.rolling(126, min_periods=60).sum()
    mom_12m  = chg_1d.rolling(252, min_periods=120).sum()
    mom_63d  = chg_1d.rolling(63,  min_periods=21).mean()
    dev_ma_3m  = level - level.rolling(63,  min_periods=21).mean()
    dev_ma_12m = level - level.rolling(252, min_periods=120).mean()
    rvol_20d   = chg_1d.rolling(20, min_periods=5).std()
    vol_mu     = rvol_20d.rolling(252, min_periods=60).mean()
    vol_sd     = rvol_20d.rolling(252, min_periods=60).std()
    vol_med    = rvol_20d.rolling(252, min_periods=60).median()
    vol_regime = (rvol_20d > vol_med).astype(float)
    vol_zscore = ((rvol_20d - vol_mu) / (vol_sd + 1e-9)).clip(-5, 5)
    lv_mu      = level.rolling(504, min_periods=120).mean()
    lv_sd      = level.rolling(504, min_periods=120).std()
    lv_zscore  = ((level - lv_mu) / (lv_sd + 1e-9)).clip(-5, 5)
    carry_roll = carry_roll_spread(sec, df, country, maturity)

    def _slope(lm, sm):
        lc, sc = f'{country}_{lm}', f'{country}_{sm}'
        if lc in df.columns and sc in df.columns:
            return df[lc] - df[sc]
        return pd.Series(np.nan, index=df.index)

    sofr_mom_1m = (df['SOFR_10Y'].diff(1).rolling(21, min_periods=10).sum()
                   if 'SOFR_10Y' in df.columns
                   else pd.Series(np.nan, index=df.index))

    base = {
        **_tlags,
        'carry_roll'         : carry_roll,
        'mom_63d'            : mom_63d,
        'mom_1m'             : mom_1m,
        'mom_3m'             : mom_3m,
        'mom_6m'             : mom_6m,
        'mom_12m'            : mom_12m,
        'dev_ma_3m'          : dev_ma_3m,
        'dev_ma_12m'         : dev_ma_12m,
        'vol_20d'            : rvol_20d,
        'vol_regime'         : vol_regime,
        'vol_zscore'         : vol_zscore,
        'level_zscore'       : lv_zscore,
        'slope_10_1'         : _slope('10Y', '1Y'),
        'slope_10_5'         : _slope('10Y', '5Y'),
        'slope_5_1'          : _slope('5Y',  '1Y'),
        'sofr_mom_1m'        : sofr_mom_1m,
        # country IBOR (contemporaneous)
        'ibor_level'         : _get(f'{country}_ibor_level'),
        'ibor_chg_1d'        : _get(f'{country}_ibor_chg_1d'),
        'ibor_mom_1m'        : _get(f'{country}_ibor_mom_1m'),
        'ibor_term_slope'    : _get(f'{country}_ibor_term_slope'),
        'swap_ibor_basis'    : level - _get(f'{country}_ibor_level'),
        # global vol (shift(1) baked into precomputed columns)
        'move_level'         : _get('move_level'),
        'move_zscore'        : _get('move_zscore'),
        'vix_level'          : _get('vix_level'),
        'vix_zscore'         : _get('vix_zscore'),
        'v2x_level'          : _get('v2x_level'),
        'vxn_level'          : _get('vxn_level'),
        'ovx_level'          : _get('ovx_level'),
        'gvz_level'          : _get('gvz_level'),
        # equity (shift(1))
        'spx_mom_1m'         : _get('spx_mom_1m'),
        'spx_mom_3m'         : _get('spx_mom_3m'),
        'sx5e_mom_1m'        : _get('sx5e_mom_1m'),
        'sx5e_mom_3m'        : _get('sx5e_mom_3m'),
        'mxwo_mom_1m'        : _get('mxwo_mom_1m'),
        'ndx_mom_1m'         : _get('ndx_mom_1m'),
        # commodities (shift(1))
        'oil_mom_1m'         : _get('oil_mom_1m'),
        'oil_mom_3m'         : _get('oil_mom_3m'),
        'copper_mom_1m'      : _get('copper_mom_1m'),
        'opec_prod'          : _get('opec_prod'),
        'natgas_mom_1m'      : _get('natgas_mom_1m'),
        # breakeven + credit (shift(1))
        'be5y_level'         : _get('breakeven_5Y_level'),
        'be10y_level'        : _get('breakeven_10Y_level'),
        'be_slope'           : _get('breakeven_slope'),
        'be5y_chg_1m'        : _get('breakeven_5Y_chg_1m'),
        'ig_spread'          : _get('ig_spread'),
        'hy_spread'          : _get('hy_spread'),
        'ig_chg_1m'          : _get('ig_chg_1m'),
        'hy_chg_1m'          : _get('hy_chg_1m'),
        # monthly macro (ffill+shift(1))
        'cpi_yoy'            : _get('cpi_yoy'),
        'pce_core_chg'       : _get('pce_core_chg'),
        'nfp_change'         : _get('nfp_change'),
        'pmi_us'             : _get('pmi_us'),
        'pmi_eur'            : _get('pmi_eur'),
        'unemp_eur'          : _get('unemp_eur'),
        'cpi_nsa_level'      : _get('cpi_nsa_level'),
        'pce_core_level'     : _get('pce_core_level'),
        # extra IBOR tenors (contemporaneous)
        'eur_3m'             : _get('eur_3m'),
        'eur_1w'             : _get('eur_1w'),
        'nor_ibor_1m'        : _get('nor_ibor_1m'),
        'nor_ibor_6m'        : _get('nor_ibor_6m'),
    }

    # Cross-market swap rates — ALL other swap cols, shift(1)
    # feature[t] = other_swap_close[t-1], safe for all timezones
    xm = {}
    other_swaps = sorted(c for c in df.columns if _SWAP_PAT.match(c) and c != sec)
    for col in other_swaps:
        fn = 'xm_' + re.sub(r'[^a-zA-Z0-9]', '_', col).strip('_')
        xm[fn + '_lv'] = df[col].shift(1)
        xm[fn + '_ch'] = df[col].diff(1).shift(1)

    # ── Norway-specific features — gated to country=='NOR' (Part B) ──────────
    # Read precomputed nor_* columns (timing baked in by add_norway_features_v4)
    # plus the swap-govvie spread for the security's own tenor (contemporaneous).
    nor = {}
    if country == 'NOR':
        for fn in sorted(c for c in df.columns if c.startswith('nor_')):
            nor[fn] = df[fn]
        gcol = f'nor_govt_{maturity.lower()}'
        nor['swap_govt_spread'] = ((level - df[gcol]) if gcol in df.columns
                                   else pd.Series(np.nan, index=df.index))

    return pd.DataFrame({**base, **xm, **nor}, index=df.index)


print('_features_for_security_v4 defined')
print(f'  target lags: {TARGET_LAGS} (chg_lag1..chg_lag{TARGET_LAGS})')
print(f'  base features: ~{63 + TARGET_LAGS}  +  cross-market: ~2 × (|UNIVERSE|-1)')

### Build the modelling panel
Stack the per-security features into one long panel and attach the forecasting target — the h-day-ahead change in the swap level.

In [ ]:
META_COLS = {'date', 'security', 'y', 'level'}


def build_panel_v4(df_raw, securities, h):
    """Stacked per-security panel. Target: outright h-day change (not residual)."""
    rows = []
    for sec in securities:
        if sec not in df_raw.columns:
            continue
        country, maturity = sec.rsplit('_', 1)
        level = df_raw[sec]
        feats = _features_for_security_v4(sec, df_raw, country, maturity)
        y     = level.shift(-h) - level
        sec_df             = feats.reset_index().rename(columns={'index': 'date',
                                                                  df_raw.index.name or 'Date': 'date'})
        # handle named index
        if 'date' not in sec_df.columns:
            sec_df.insert(0, 'date', feats.index)
        sec_df['security'] = sec
        sec_df['y']        = y.values
        sec_df['level']    = level.values
        rows.append(sec_df)

    if not rows:
        return pd.DataFrame()
    panel = pd.concat(rows, ignore_index=True)
    panel = panel.dropna(subset=['y'])
    panel = panel.sort_values(['date', 'security']).reset_index(drop=True)
    return panel


print('build_panel_v4 defined')

### Apply the macro/global features
Run the macro feature builder on the raw panel and report which columns were added. The builder derives columns from raw source series, so re-running recomputes identical values rather than double-applying.

In [ ]:
_cols_before = set(df_raw.columns)
t0 = time.time()
df_raw = add_macro_features_v4(df_raw)
print(f'df_raw augmented in {time.time()-t0:.1f}s  →  shape: {df_raw.shape}')

new_cols = sorted(c for c in df_raw.columns if c not in _cols_before)
print(f'\nDerived macro/global columns added ({len(new_cols)}):')
print(f'  {"Column":<35s}  {"n_obs":>6s}')
print('  ' + '-' * 44)
for c in new_cols:
    n = df_raw[c].notna().sum()
    print(f'  {c:<35s}  {n:6d}')


# Norway nor_* columns were added in Part B (above) and are preserved here.
_n_nor = sum(1 for c in df_raw.columns if c.startswith('nor_'))
print(f'\nNorway nor_* feature columns present after augmentation: {_n_nor}')

### Inspect one security's features
Print the feature matrix for the pilot security with a timing annotation per column, as an eyeball check that no exogenous feature is contemporaneous or forward-looking.

In [ ]:
# ── Step 0a: Print sample feature matrix with lags ───────────────────────────
# For one security, show all feature columns with their timing/lag annotation.
# Eyeball check: nothing contemporaneous-or-future for exogenous series.

_c, _m    = SMOKE_SEC.rsplit('_', 1)
_feat_df  = _features_for_security_v4(SMOKE_SEC, df_raw, _c, _m)
# Sample the last date where THIS security actually trades (own-security features
# populated). The last calendar row lands past the swap's data end, where only
# ffill-macro / cross-market features survive and every own column is NaN.
_last_idx = df_raw[SMOKE_SEC].dropna().index[-1]
_row      = _feat_df.loc[_last_idx]

print(f'=== Feature matrix for {SMOKE_SEC} ===')
print(f'  Shape: {_feat_df.shape[0]} dates × {_feat_df.shape[1]} features')
print(f'  Sample date: {_last_idx}')
print()

# Timing annotations
def _timing(fname):
    if fname.startswith('chg_lag'):
        return 'backward AR lag (chg_1d shifted k days; own security)'
    if fname in ('carry_roll','mom_63d','mom_1m','mom_3m','mom_6m','mom_12m',
                 'dev_ma_3m','dev_ma_12m','vol_20d','vol_regime','vol_zscore',
                 'level_zscore','slope_10_1','slope_10_5','slope_5_1'):
        return 'contemporaneous (own security daily close)'
    if fname in ('ibor_level','ibor_chg_1d','ibor_mom_1m','ibor_term_slope',
                 'swap_ibor_basis','sofr_mom_1m','eur_3m','eur_1w',
                 'nor_ibor_1m','nor_ibor_6m'):
        return 'contemporaneous (IBOR / same-session fix)'
    if any(fname.startswith(p) for p in ('move_','vix_','v2x_','vxn_','rvx_',
                                          'ovx_','gvz_')):
        return 'shift(1): vol index, prior close'
    if any(fname.startswith(p) for p in ('spx_','sx5e_','mxwo_','ndx_')):
        return 'shift(1): equity close, prior day'
    if any(fname.startswith(p) for p in ('oil_','copper_','opec_','natgas_')):
        return 'shift(1): commodity close, prior day'
    if any(fname.startswith(p) for p in ('be5y_','be10y_','be_slope','ig_','hy_')):
        return 'shift(1): TIPS/credit market, prior day'
    if any(fname.startswith(p) for p in ('cpi_','pce_','nfp_','pmi_','unemp_')):
        return 'ffill+shift(1): monthly release, prior value'
    if fname.startswith('xm_'):
        return 'shift(1): cross-market swap, prior close'
    if fname.startswith('nor_'):
        if any(fname.startswith(p) for p in ('nor_eurnok', 'nor_usdnok', 'nor_i44')):
            return 'shift(1): NOK FX / krone index, prior close'
        if any(fname.startswith(p) for p in ('nor_kpi', 'nor_kpijae', 'nor_unemp')):
            return 'ffill+shift(1): SSB monthly release, prior value'
        return 'contemporaneous (NOK official / administered same-session)'
    if fname == 'swap_govt_spread':
        return 'contemporaneous (swap − govvie, same-session)'
    return '?'

n_xm   = sum(1 for f in _row.index if f.startswith('xm_'))
n_base = len(_row) - n_xm
print(f'  {"Feature":<42s}  {"Value":>10s}  Timing')
print('  ' + '-' * 90)
for feat, val in _row.items():
    if feat.startswith('xm_') and n_xm > 6:
        continue  # print summary line instead of all cross-market
    val_str = f'{val:.4f}' if pd.notna(val) else 'NaN'
    print(f'  {feat:<42s}  {val_str:>10s}  {_timing(feat)}')
if n_xm > 6:
    non_nan_xm = sum(1 for f,v in _row.items() if f.startswith('xm_') and pd.notna(v))
    print(f'  {"[xm_* cross-market features]":<42s}  '
          f'{non_nan_xm:>10d} non-NaN  shift(1): cross-market swap, prior close')
print(f'\n  Base features: {n_base}   Cross-market features: {n_xm}   Total: {len(_row)}')
print()
print('[LEAKAGE CHECK] All exogenous features (VIX, equity, commodity, credit, cross-market)')
print('  are lagged 1 period. Own-security features are contemporaneous (daily close).')
print('  Monthly macro series were already ffill+shift(1) in v3; unchanged.')

### Leakage audit
Independently re-derive each feature from data available only up to time *t* and compare it against the stored panel value, so any look-ahead leak fails loudly before the heavy runs.

**Scope note.** The audit fully re-derives the own-curve and cross-market features point-in-time. The precomputed macro/global columns are validated as already-lagged stored values rather than being re-derived independently — a known scope boundary, not a bug.

In [ ]:
# ── Step 0b: Leakage audit on full expanded feature set ──────────────────────
# Recompute features from df_raw.loc[<=t] and compare with stored panel values.
# Audits ALL columns including cross-market swap feats.

def leakage_audit_v4(panel, df_raw, n_samples=30, rng_seed=42, skip_first=500):
    _META     = {'date', 'security', 'y', 'level'}
    feat_cols = [c for c in panel.columns if c not in _META]

    eligible = panel.iloc[skip_first:].copy()
    sample   = eligible.sample(n=min(n_samples, len(eligible)), random_state=rng_seed)

    TOL          = 1e-6
    fail_counts  = {c: 0 for c in feat_cols}
    check_counts = {c: 0 for c in feat_cols}

    for _, row in sample.iterrows():
        t       = row['date']
        sec     = row['security']
        country, maturity = sec.rsplit('_', 1)
        past = df_raw.loc[df_raw.index <= t]
        if sec not in past.columns or past[sec].isna().all():
            continue
        feats_at_t = _features_for_security_v4(sec, past, country, maturity).iloc[-1]

        for col in feat_cols:
            stored = row[col]
            recomp = feats_at_t.get(col, np.nan)
            check_counts[col] += 1
            if pd.isna(stored) and pd.isna(recomp):
                continue
            if pd.isna(stored) != pd.isna(recomp):
                fail_counts[col] += 1
                continue
            if abs(float(stored) - float(recomp)) > TOL:
                fail_counts[col] += 1

    n_fail = sum(1 for c in feat_cols if fail_counts[c] > 0)
    n_skip = sum(1 for c in feat_cols if check_counts[c] == 0)
    n_pass = len(feat_cols) - n_fail - n_skip

    print(f'=== Leakage audit v4 — {len(feat_cols)} features, '
          f'{n_samples} sample rows ===\n')

    for col in feat_cols:
        if fail_counts[col] > 0:
            n = check_counts[col]
            print(f'  FAIL  {col}  {n-fail_counts[col]}/{n} OK  '
                  f'← {fail_counts[col]} mismatch(es)')

    print(f'\n  Summary: {n_pass} PASS / {n_fail} FAIL / {n_skip} SKIP')
    if n_fail == 0:
        print(f'\n[PASS] All {n_pass} checked features leakage-free.')
        print(f'  VIX/MOVE/equity/commodity: shift(1) in add_macro_features_v4.')
        print(f'  Cross-market swap rates: shift(1) in _features_for_security_v4.')
    else:
        print(f'\n[FAIL] {n_fail} feature(s) have mismatches — fix before full run.')

    return n_fail == 0


_pilot_panel = build_panel_v4(df_raw, [SMOKE_SEC], H)
print(f'Pilot panel for audit: {_pilot_panel.shape}  '
      f'({_pilot_panel.shape[1]-len(META_COLS)} feature cols)')
print()
_audit_ok = leakage_audit_v4(_pilot_panel, df_raw, n_samples=30)
assert _audit_ok, 'Leakage audit failed — fix before proceeding'

## Metrics contract — shared long-CSV schema

Every results row — walk-forward **and** block-CV/LORO, **train** and **oos** — is produced by the
single function `compute_metrics_row(...)` and written to **`v5_metrics_long.csv`**. The pooled
notebook must emit the **identical** shared columns (it may add pooled-only columns; it must never
rename or drop a shared one). One row per `(security, tenor, horizon, fold/block, scheme, sample)`.

**Identifier / provenance columns**

| column | dtype | meaning |
|---|---|---|
| `notebook` | str | `'v5'` |
| `run_ts` | str | ISO timestamp of the run |
| `model_kind` | str | `'per_security'` here; `'pooled'` in the pooled notebook |
| `is_pooled` | bool | `False` here, `True` in the pooled notebook |
| `validation_scheme` | str | `'walk_forward'` \| `'block_cv'` \| `'loro'` |
| `target_kind` | str | `'level_change'` (default) \| `'excess_hpr'` (optional long-horizon target) |
| `security` | str | e.g. `NOR_10Y` |
| `country` | str | e.g. `NOR` |
| `tenor` | str | e.g. `10Y` |
| `horizon` | int | h (business days) |
| `fold` | str | fold/block name (e.g. `Hiking`, `Block_ZIRP`) |
| `regime` | str | `GFC`\|`ZIRP`\|`COVID`\|`Hiking` |
| `sample` | str | `'train'` \| `'oos'` |
| `config_hash` | str | hash of the frozen config + feature spec |
| `feature_count` | int | #features actually fed to HTBoost (post-PCA) |

**Metric columns** (random-walk benchmark forecasts the change as `ŷ_RW = 0`, so `mse_rw = mean(y²)`)

| column | dtype | meaning |
|---|---|---|
| `n_obs` | int | scored observations |
| `dir_acc` | float | directional accuracy, `mean(sign(y)==sign(ŷ))` |
| `r2_raw` | float | `sklearn.r2_score(y, ŷ)` |
| `mse_model` | float | `mean((y−ŷ)²)` |
| `mse_rw` | float | `mean(y²)` |
| `ct_r2_oos` | float | **Campbell–Thompson** OOS-R² `= 1 − mse_model/mse_rw` |
| `cw_stat`,`cw_pval` | float | **Clark–West** vs RW (HAC/Newey–West, lag≈H−1), one-sided (model better) |
| `dmh_stat`,`dmh_pval` | float | **Diebold–Mariano–Harvey** vs RW (HLN small-sample corr., HAC lag≈H−1), one-sided |
| `pt_stat`,`pt_pval` | float | **Pesaran–Timmermann** directional, one-sided |
| `binom_pval` | float | one-sided binomial vs 0.5 |
| `n_eff` | int | effective sample size for overlapping returns `≈ n/H` (binomial/PT caveat) |

**MTC columns** (added per family, **never pooled across families**)

| column | dtype | meaning |
|---|---|---|
| `reject_bonferroni` | bool | survives Bonferroni within its MTC family |
| `reject_fdr_bh` | bool | survives BH-FDR within its MTC family |
| `mtc_N` | int | family size used |
| `mtc_family` | str | `walk_forward:{horizon×tenor×regime}` or `{scheme}:{horizon×tenor×block}` |

Walk-forward and block-CV/LORO are corrected in **separate MTC families** (they test
forecastability vs learnability — different hypotheses).

In [ ]:
# ── Metrics contract — import the SHARED single source of truth ──────────────────────
# Instead of re-defining the metric functions, we IMPORT htb_metrics.py — the exact
# module the HTBoost per-security and pooled notebooks use. Every linear row is therefore
# produced by byte-identical code: Clark–West / Diebold–Mariano–Harvey (HAC lag = H−1),
# Pesaran–Timmermann, Campbell–Thompson OOS R², directional accuracy, and the binomial
# test on n_eff ≈ n/H. SHARED_COLS is the frozen pooled-comparability schema; the linear
# tables concatenate with v5_metrics_long.csv with zero reconciliation.
import importlib
import htb_metrics
importlib.reload(htb_metrics)
from htb_metrics import (compute_metrics_row, SHARED_COLS, config_hash,
                         clark_west, dm_harvey, pesaran_timmermann, _score)

# Override provenance tags so every emitted row is stamped notebook='linear_v5'.
htb_metrics.NOTEBOOK_TAG = NOTEBOOK_TAG
htb_metrics.RUN_TS       = RUN_TS

print('Shared metrics contract loaded from htb_metrics.py')
print(f'  notebook tag = {htb_metrics.NOTEBOOK_TAG!r}')
print(f'  SHARED_COLS ({len(SHARED_COLS)}): {SHARED_COLS}')
print('  NOTE: htb_metrics defaults model_kind="per_security"; the linear scorer '
      'OVERRIDES it to "OLS"/"ElasticNet" on every row (schema otherwise untouched).')

### Cross-market PCA (fit on training rows only)
Compress the large cross-market `xm_*` block to a handful of principal components, fitting the standardizer and PCA on training rows only and applying them to the test rows. Living in the fit path (not the feature builder) keeps the leakage audit valid.

In [ ]:
# ── PCA compression of the cross-market xm_* block — FIT ON TRAIN ONLY ──
# Splits the feature matrix into xm_* (cross-market) vs the rest. Standardises and
# PCA-fits ONLY on the training rows, then transforms train AND test with the SAME
# fitted objects, so test data never touches the PCA fit. The xm_* columns are
# replaced by k components (xmpca_01..xmpca_kk). Pre-committed rule:
#   k = smallest #components explaining ≥ XM_PCA_VAR of TRAIN variance, capped at XM_PCA_KMAX.
# This is in the FIT PATH — deliberately NOT in _features_for_security_v4 (the leakage
# audit recomputes that function from df_raw.loc[<=t] and cannot validate a fold-fit
# transform; putting PCA there would either break the audit or leak).

def _xm_split(cols):
    xm   = [c for c in cols if c.startswith('xm_')]
    rest = [c for c in cols if not c.startswith('xm_')]
    return xm, rest


def fit_transform_xm_pca(x_tr_df, x_te_df, var_target=XM_PCA_VAR, kmax=XM_PCA_KMAX,
                         enable=XM_PCA_ENABLE):
    """Return (x_tr_new, x_te_new, info). info: applied, k, n_xm_in, evr_cum_k, var_target.
    Round-trips through walk-forward: fit on x_tr only, apply to both."""
    cols = list(x_tr_df.columns)
    xm, rest = _xm_split(cols)
    info = {'applied': False, 'k': 0, 'n_xm_in': len(xm),
            'evr_cum_k': np.nan, 'var_target': var_target}
    if (not enable) or len(xm) < 2:
        return x_tr_df, x_te_df, info

    def _to_arr(df):
        return (df[xm].replace([np.inf, -np.inf], np.nan)
                .astype(np.float64).to_numpy())

    Xtr, Xte = _to_arr(x_tr_df), _to_arr(x_te_df)

    # Standardise on TRAIN moments computed over observed (non-NaN) entries only.
    mu = np.nanmean(Xtr, axis=0)
    sd = np.nanstd(Xtr, axis=0)
    sd = np.where(np.isfinite(sd) & (sd > 0), sd, 1.0)   # guard zero/NaN-variance cols
    mu = np.where(np.isfinite(mu), mu, 0.0)

    Ztr = (Xtr - mu) / sd
    Zte = (Xte - mu) / sd

    # Zero-fill AFTER standardising = training-mean imputation (text §6.5).
    Ztr = np.nan_to_num(Ztr, nan=0.0, posinf=0.0, neginf=0.0)
    Zte = np.nan_to_num(Zte, nan=0.0, posinf=0.0, neginf=0.0)

    kfit = min(len(xm), Xtr.shape[0])
    pca_full = PCA(n_components=kfit).fit(Ztr)         # fit on TRAIN only
    cum = np.cumsum(pca_full.explained_variance_ratio_)
    k = int(np.searchsorted(cum, var_target) + 1)
    k = max(1, min(k, kmax, kfit))

    pca = PCA(n_components=k).fit(Ztr)
    Ctr, Cte = pca.transform(Ztr), pca.transform(Zte)  # SAME fitted PCA on both
    comp_names = [f'xmpca_{i+1:02d}' for i in range(k)]

    tr_new = pd.concat(
        [x_tr_df[rest].reset_index(drop=True),
         pd.DataFrame(Ctr, columns=comp_names)], axis=1)
    tr_new.index = x_tr_df.index
    te_new = pd.concat(
        [x_te_df[rest].reset_index(drop=True),
         pd.DataFrame(Cte, columns=comp_names)], axis=1)
    te_new.index = x_te_df.index

    info.update({'applied': True, 'k': k,
                 'evr_cum_k': float(cum[k - 1])})
    return tr_new, te_new, info


print('PCA fit-path helper defined: fit_transform_xm_pca (fit on TRAIN rows only)')

### Feature taxonomy
Tag every feature column to a bucket (curve, momentum, vol, macro, credit, cross-market, Norway, carry/roll) for the feature-importance report and the macro-vs-curve comparison.

In [ ]:
# ── feature taxonomy — every column tagged to a bucket ──────────────────
# Buckets: curve, momentum, vol, macro, credit, cross_market, norway, carry_roll.
# Reused for feature-importance aggregation (report) and the macro-vs-curve
# comparison tied to Bianchi et al. The PCA components (xmpca_*) inherit the
# cross_market bucket, so importance attributed to compressed cross-market info is
# still bucketed correctly.

def bucket_feature(name):
    n = str(name)
    if n == 'carry_roll':
        return 'carry_roll'
    if n.startswith('xm_') or n.startswith('xmpca_'):
        return 'cross_market'
    if n.startswith('nor_') or n == 'swap_govt_spread':
        return 'norway'
    if n.startswith('ig_') or n.startswith('hy_'):
        return 'credit'
    if (n.startswith(('vol_', 'move_', 'vix_', 'v2x_', 'vxn_', 'ovx_', 'gvz_', 'rvx_'))):
        return 'vol'
    if n.startswith(('mom_', 'chg_lag')):
        return 'momentum'
    if (n in ('level_zscore', 'swap_ibor_basis') or
            n.startswith(('slope_', 'dev_ma_'))):
        return 'curve'
    # everything else: rates/macro/equity/commodity/breakeven exogenous → macro
    return 'macro'


# Build the taxonomy on the smoke-security panel feature columns (pre-PCA names;
# xm_* are listed individually and also summarised as their PCA collapse).
_tax_panel = build_panel_v4(df_raw, [SMOKE_SEC], H)
_tax_feats = [c for c in _tax_panel.columns if c not in META_COLS]
feature_taxonomy = pd.DataFrame({
    'feature': _tax_feats,
    'bucket':  [bucket_feature(c) for c in _tax_feats],
})
feature_taxonomy.to_csv(f'{OUT_DIR}/v5_feature_taxonomy.csv', index=False)

print('=== feature taxonomy ===')
_counts = feature_taxonomy['bucket'].value_counts().sort_index()
print(f'  {"bucket":<14s} {"#features (pre-PCA)":>20s}')
print('  ' + '-' * 36)
for b, c in _counts.items():
    print(f'  {b:<14s} {c:>20d}')
print(f'\n  Total feature columns (pre-PCA): {len(_tax_feats)}')
print(f'  cross_market (xm_*) collapse to ≤ {XM_PCA_KMAX} PCA comps in the fit path.')
print(f'  Saved: {OUT_DIR}/v5_feature_taxonomy.csv')

# Sanity: confirm Norway features are present & bucketed
_nor_feats = feature_taxonomy[feature_taxonomy['bucket'] == 'norway']['feature'].tolist()
print(f'\n  Norway-bucket features wired for {SMOKE_SEC}: {len(_nor_feats)}')
assert len(_nor_feats) > 0, 'No Norway features bucketed — check Part B wiring'
for c in _nor_feats[:40]:
    print(f'    {c}')

### Linear estimators (OLS + Elastic Net)

In [ ]:
# ── Linear estimators — the ONLY cell that differs in substance from HTBoost ─────────
# This replaces the HTBoost juliacall bridge. Both estimators consume the SAME
# PCA-compressed feature matrix the HTBoost notebook builds (the PCA happens upstream in
# _fit_score_rows, fit on TRAIN rows only). Standardisation is fit on TRAIN rows and
# applied to test rows INSIDE the fit path — required for a penalised model and kept
# leakage-free. Standardising does not change OLS predictions/R²/DirAcc; it makes the
# coefficient magnitudes comparable across features for the taxonomy report.

def prepare_x_lin(df):
    """Deduplicate columns and coerce to finite float64. Column NAMES are preserved
    (unlike the Julia bridge) so coefficients map cleanly onto the feature taxonomy."""
    df = df.copy()
    # de-duplicate any repeated column labels (paranoia; mirrors the v5 bridge)
    seen, cols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c] += 1; cols.append(f'{c}_{seen[c]}')
        else:
            seen[c] = 0; cols.append(c)
    df.columns = cols
    return df.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float64)


def fit_linear(y_train, x_train_df, x_test_df, model_kind, H):
    """Fit one linear estimator with TRAIN-only standardisation. Returns
    (output, yhat_train, yhat_test_or_None, info, coef_dict).

    info carries the chosen Elastic-Net penalty (alpha, l1_ratio); for OLS these are NaN.
    coef_dict maps feature name → standardised-space coefficient (for the taxonomy
    coefficient-mass report). H is accepted for signature parity with the HTBoost bridge
    (linear fits do not use the overlap directly; the OVERLAP=H−1 purge is applied
    upstream when the train/test windows are cut)."""
    x_tr = prepare_x_lin(x_train_df)
    feat_names = list(x_tr.columns)
    y_train = np.asarray(y_train, dtype=float)

    scaler = StandardScaler().fit(x_tr.to_numpy())        # TRAIN mean/std only
    Z_tr = scaler.transform(x_tr.to_numpy())

    info = {'model_kind': model_kind, 'n_features': len(feat_names),
            'alpha': np.nan, 'l1_ratio': np.nan, 'n_nonzero': len(feat_names)}

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')                   # silence ConvergenceWarning
        if model_kind == 'OLS':
            model = LinearRegression(fit_intercept=True).fit(Z_tr, y_train)
        elif model_kind == 'ElasticNet':
            # alpha + l1_ratio selected by internal TimeSeriesSplit CV on TRAIN rows only.
            model = ElasticNetCV(
                l1_ratio=EN_L1_GRID, n_alphas=EN_N_ALPHAS,
                cv=TimeSeriesSplit(n_splits=EN_CV_SPLITS),
                max_iter=EN_MAX_ITER, tol=EN_TOL, fit_intercept=True,
                random_state=SEED, selection='cyclic', n_jobs=1).fit(Z_tr, y_train)
            info['alpha']     = float(model.alpha_)
            info['l1_ratio']  = float(model.l1_ratio_)
            info['n_nonzero'] = int(np.sum(np.abs(model.coef_) > 1e-12))
        else:
            raise ValueError(model_kind)

    yhat_tr = np.asarray(model.predict(Z_tr), dtype=float)
    yhat_te = None
    if x_test_df is not None:
        x_te = prepare_x_lin(x_test_df)
        # align test columns to the train design matrix (defensive; order must match)
        x_te = x_te.reindex(columns=feat_names, fill_value=0.0)
        yhat_te = np.asarray(model.predict(scaler.transform(x_te.to_numpy())), dtype=float)

    coef = dict(zip(feat_names, np.asarray(model.coef_, dtype=float)))
    output = {'model': model, 'scaler': scaler, 'feat_names': feat_names,
              'intercept': float(model.intercept_), 'coef': coef}
    return output, yhat_tr, yhat_te, info, coef


print('Linear estimators defined: fit_linear (OLS, ElasticNet) + prepare_x_lin')
print('  ElasticNet penalty selected by internal TimeSeriesSplit CV on TRAIN rows only.')

### Fit→(train+OOS) scorer and walk-forward runner

In [ ]:
# ── shared fit→(train+oos) scorer, reused by walk-forward AND block-CV ──
# Structure is IDENTICAL to v5's _fit_score_rows / run_security_v5: PCA-compress (fit on
# train rows only) → fit → emit one TRAIN row and one OOS row in the SHARED schema. The
# only change is the estimator call (fit_linear instead of fit_htboost_v5) and that
# model_kind is threaded so OLS and ElasticNet rows are produced by the same code path.
FEAT_SPEC = f'pca_var{XM_PCA_VAR}_kmax{XM_PCA_KMAX}_tl{TARGET_LAGS}'


def _meta_base(sec, H, cfg, scheme, fold, regime, feature_count,
               target_kind='level_change'):
    country, tenor = sec.rsplit('_', 1)
    return dict(validation_scheme=scheme, target_kind=target_kind,
                security=sec, country=country, tenor=tenor, fold=fold, regime=regime,
                config_hash=config_hash(cfg, extra=FEAT_SPEC), feature_count=feature_count)


def _fit_score_rows(tr, te, fc, H, model_kind, sec, scheme, fold, regime,
                    target_kind='level_change'):
    """PCA-compress (fit on tr only) → fit linear model → TRAIN row + OOS row.
    Returns (rows, fitinfo). fitinfo carries the standardised coefficients and the
    Elastic-Net penalty for the coefficient/sparsity and appendix reports."""
    cfg = lin_cfg(model_kind)
    x_tr, x_te, pca = fit_transform_xm_pca(tr[fc], te[fc])
    feat_count = x_tr.shape[1]
    out, yhat_tr, yhat_te, info, coef = fit_linear(
        tr['y'].to_numpy(float), x_tr, x_te, model_kind, H)
    base = _meta_base(sec, H, cfg, scheme, fold, regime, feat_count, target_kind)
    row_tr = compute_metrics_row(tr['y'].to_numpy(float), yhat_tr, H,
                                 {**base, 'sample': 'train'})
    row_te = compute_metrics_row(te['y'].to_numpy(float), yhat_te, H,
                                 {**base, 'sample': 'oos'})
    for r in (row_tr, row_te):
        r['model_kind'] = model_kind                       # override shared default
        r['pca_k']      = pca['k']
        r['alpha']      = info['alpha']                    # EN penalty (NaN for OLS)
        r['l1_ratio']   = info['l1_ratio']
        r['n_nonzero']  = info['n_nonzero']
    fitinfo = {'security': sec, 'tenor': tenor_of(sec), 'horizon': int(H), 'fold': fold,
               'regime': regime, 'scheme': scheme, 'model_kind': model_kind,
               'feature_count': feat_count, 'pca_k': pca['k'], 'coef': coef,
               'alpha': info['alpha'], 'l1_ratio': info['l1_ratio'],
               'n_nonzero': info['n_nonzero']}
    return [row_tr, row_te], fitinfo


def tenor_of(sec):
    return sec.rsplit('_', 1)[1]


def run_security_lin(sec, df_raw, H, model_kind, verbose=False):
    """Walk-forward (causal, expanding window) across all folds for one security at
    horizon H, for ONE estimator. Emits a TRAIN and an OOS row per populated fold, with
    the one-sided OVERLAP=H−1 purge — IDENTICAL fold geometry to run_security_v5."""
    if sec not in df_raw.columns:
        return [], []
    panel = build_panel_v4(df_raw, [sec], H)
    if len(panel) == 0:
        return [], []
    fc = [c for c in panel.columns if c not in META_COLS]
    rows, fits = [], []
    for fold_name, test_start, test_end, regime in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(test_start), pd.Timestamp(test_end)
        purge_ts = ts_ts - pd.tseries.offsets.BDay(H - 1)          # OVERLAP = H-1
        tr = panel[panel['date'] < purge_ts].copy()
        te = panel[(panel['date'] >= ts_ts) & (panel['date'] <= te_ts)].copy()
        if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
            if verbose:
                print(f'  {fold_name}: skip (train={len(tr)}, test={len(te)})')
            continue
        try:
            r, fi = _fit_score_rows(tr, te, fc, H, model_kind, sec, 'walk_forward',
                                    fold_name, regime)
        except Exception as e:
            if verbose:
                print(f'  {fold_name}: FAILED — {repr(e)[:80]}')
            continue
        rows.extend(r); fits.append(fi)
        if verbose:
            oos = r[1]
            print(f'  {fold_name:<14s} train DA={r[0]["dir_acc"]:.3f}  '
                  f'OOS DA={oos["dir_acc"]:.3f}  CT-R²={oos["ct_r2_oos"]:+.4f}  n={oos["n_obs"]}')
    return rows, fits


print('Walk-forward linear runner defined: run_security_lin (per estimator; '
      'emits train+oos rows per fold, plus coefficient/penalty fitinfo)')

### Smoke test (both estimators, all folds)

In [ ]:
# ── Smoke test (linear) — one tenor (SMOKE_SEC), one horizon (H), all folds, BOTH models
# Mirrors the v5 smoke: PCA-compress xm_* on TRAIN rows only, fit, score train AND OOS,
# and REPORT the in-sample magnitude and the OOS-vs-0.50 distance — never tuning to move
# them. Tripwires test PLUMBING (PCA wired and non-degenerate, curve features survive
# into the model matrix, predictions finite and non-constant, a real train→OOS gap), not
# a fit magnitude. In-sample fit is a diagnostic only.
assert SMOKE_SEC in df_raw.columns

# ── (a) PCA round-trip / plumbing check on the Hiking fold (shared with HTBoost) ──────
_panel = build_panel_v4(df_raw, [SMOKE_SEC], H)
_fc    = [c for c in _panel.columns if c not in META_COLS]
_ts, _te_end = pd.Timestamp(WF_FOLDS[-1][1]), pd.Timestamp(WF_FOLDS[-1][2])
_purge = _ts - pd.tseries.offsets.BDay(H - 1)
_tr = _panel[_panel['date'] < _purge].copy()
_te = _panel[(_panel['date'] >= _ts) & (_panel['date'] <= _te_end)].copy()
assert len(_tr) >= MIN_TRAIN_OBS and len(_te) >= MIN_TEST_OBS
_x_tr, _x_te, _pca = fit_transform_xm_pca(_tr[_fc], _te[_fc])
_feat_count = _x_tr.shape[1]

_xmpca_cols = [c for c in _x_tr.columns if c.startswith('xmpca_')]
_xmpca_std  = _x_tr[_xmpca_cols].std()
print('PCA component std (train):', ' '.join(f'{s:.3g}' for s in _xmpca_std.values))
assert (_xmpca_std > 1e-9).all(), 'A xmpca_* component is constant — real PCA bug'
_curve_cols = [c for c in _x_tr.columns if bucket_feature(c) == 'curve']
assert 'level_zscore' in _x_tr.columns, 'level_zscore missing from model matrix — real bug'
assert any(c.startswith('slope_') for c in _x_tr.columns), 'no slope_* cols — real bug'
assert len(_curve_cols) >= 3, f'too few curve cols ({len(_curve_cols)}) — real bug'
print(f'\nSmoke: {SMOKE_SEC}  H={H}   features pre-PCA={len(_fc)} '
      f'(xm_*={sum(1 for f in _fc if f.startswith("xm_"))})')
print(f'  PCA applied={_pca["applied"]}  k={_pca["k"]} '
      f'(≥{_pca["var_target"]:.0%} train var → cum={_pca["evr_cum_k"]:.3f})  '
      f'features post-PCA={_feat_count}')

# Leakage plumbing: the scaler + PCA see ONLY training rows. Demonstrate that the test
# design matrix is produced by transform() (never fit) — structurally identical to v5.
_sc = StandardScaler().fit(prepare_x_lin(_x_tr).to_numpy())
_ztr_mean = np.abs(_sc.transform(prepare_x_lin(_x_tr).to_numpy()).mean(axis=0)).max()
print(f'  leakage check: train standardised mean≈{_ztr_mean:.2e} (≈0); '
      f'scaler/PCA fit on TRAIN rows only, applied to test by transform().')

# ── (b) per-estimator walk-forward smoke across all folds ────────────────────────────
smoke_fits = {}
for mk in MODEL_KINDS:
    print(f'\n=== {mk} — walk-forward smoke ({SMOKE_SEC}, H={H}) ===')
    t0 = time.time()
    rows, fits = run_security_lin(SMOKE_SEC, df_raw, H, mk, verbose=True)
    smoke_fits[mk] = (rows, fits)
    _tr_rows = [r for r in rows if r['sample'] == 'train']
    _oos_rows = [r for r in rows if r['sample'] == 'oos']
    if _oos_rows:
        _mtr = np.mean([r['dir_acc'] for r in _tr_rows])
        _moos = np.mean([r['dir_acc'] for r in _oos_rows])
        _mct = np.mean([r['ct_r2_oos'] for r in _oos_rows])
        print(f'  mean train DA={_mtr:.3f}  mean OOS DA={_moos:.3f}  '
              f'mean overfit gap={_mtr-_moos:+.3f}  mean OOS CT-R²={_mct:+.4f}')
        # plumbing asserts: finite, non-degenerate predictions; a real (≥0) train→OOS gap
        assert all(np.isfinite(r['dir_acc']) for r in _oos_rows), 'NaN OOS DirAcc'
    if mk == 'ElasticNet':
        _al = [(f['fold'], f['alpha'], f['l1_ratio'], f['n_nonzero']) for f in fits]
        print('  ElasticNet selected penalty per fold (alpha, l1_ratio, #nonzero):')
        for fold, a, l, nz in _al:
            print(f'    {fold:<14s} alpha={a:.4g}  l1_ratio={l:.2g}  nonzero={nz}/{fits[0]["feature_count"]}')
    print(f'  ({time.time()-t0:.1f}s)')

# ── (c) coefficient-mass-by-bucket spot check (Hiking fold, both models) ─────────────
print('\n=== Smoke coefficient mass by taxonomy bucket (Hiking fold) ===')
for mk in MODEL_KINDS:
    _, fits = smoke_fits[mk]
    _hik = [f for f in fits if f['fold'] == 'Hiking']
    if not _hik:
        continue
    _b = {}
    for feat, val in _hik[0]['coef'].items():
        bk = bucket_feature(feat); _b[bk] = _b.get(bk, 0.0) + abs(val)
    _tot = sum(_b.values()) or 1.0
    print(f'  {mk}:')
    for b, v in sorted(_b.items(), key=lambda kv: -kv[1]):
        print(f'    {b:<14s} |coef| share = {100*v/_tot:5.1f}%')

print('\n[PASS] linear smoke — PCA round-trip, leakage-safe standardisation, '
      'populated train+OOS rows for OLS and ElasticNet (reported, not tuned).')

### Resolve the horizon grid

In [ ]:
# ── resolve the horizon grid; gate 126/252 on data length (IDENTICAL rule to v5) ─────
# H_GRID always runs. Long horizons (126/252) are included ONLY if the data length
# supports at least one walk-forward fold for the smoke security — same gate as HTBoost,
# so the two notebooks sweep the same horizon set.

def _horizon_supported(h):
    panel = build_panel_v4(df_raw, [SMOKE_SEC], h)
    if len(panel) == 0:
        return False
    for _, ts, te, _r in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(ts), pd.Timestamp(te)
        purge = ts_ts - pd.tseries.offsets.BDay(h - 1)
        ntr = (panel['date'] < purge).sum()
        nte = ((panel['date'] >= ts_ts) & (panel['date'] <= te_ts)).sum()
        if ntr >= MIN_TRAIN_OBS and nte >= MIN_TEST_OBS:
            return True
    return False


HORIZONS = list(H_GRID)
for h in H_GRID_LONG:
    if _horizon_supported(h):
        HORIZONS.append(h)
        print(f'  long horizon H={h}: data supports it → included')
    else:
        print(f'  long horizon H={h}: insufficient data per fold → SKIPPED (logged)')
HORIZONS = sorted(set(HORIZONS))
SWEEP_TENORS = list(UNIVERSE)

print(f'\nFinal horizon grid: {HORIZONS}')
print(f'Sweep tenors ({len(SWEEP_TENORS)}): {SWEEP_TENORS}')
print(f'Estimators: {MODEL_KINDS}  (no frozen hyper-config needed — OLS has none; '
      f'ElasticNet selects its penalty internally on train rows per fold)')

### Walk-forward sweep

In [ ]:
# ── WALK-FORWARD SWEEP — model_kind × H_GRID × tenors × folds, train+OOS rows ──
# Loops the SAME (horizon × tenor × fold) grid as v5, once per estimator. Linear fits are
# fast (seconds), but the sweep is still gated by RUN_WALK_FORWARD (default OFF) per the
# null-confirmation discipline. Partial rows persisted for inspection.
wf_rows, wf_fits = [], []
_wf_failed = []

if not RUN_WALK_FORWARD:
    print('[skipped] RUN_WALK_FORWARD = False  '
          '(set RUN_FULL_SWEEP=True in the linear override cell to run the sweep)')
else:
    _t0 = time.time()
    _total = len(MODEL_KINDS) * len(HORIZONS) * len(SWEEP_TENORS)
    _i = 0
    for mk in MODEL_KINDS:
        for H_run in HORIZONS:
            for sec in SWEEP_TENORS:
                _i += 1; _ts = time.time()
                try:
                    rows, fits = run_security_lin(sec, df_raw, H_run, mk, verbose=False)
                except Exception as e:
                    print(f'  [{_i}/{_total}] {mk} H={H_run} {sec}: FAILED {repr(e)[:60]}')
                    _wf_failed.append(('walk_forward', mk, H_run, sec)); continue
                wf_rows.extend(rows); wf_fits.extend(fits)
                _oos = [r for r in rows if r['sample'] == 'oos']
                _da = ', '.join(f'{r["dir_acc"]:.3f}' for r in _oos)
                print(f'  [{_i}/{_total}] {mk:<10s} H={H_run:<3d} {sec:<10s} '
                      f'folds={len(_oos)} OOS_DA=[{_da}]  ({time.time()-_ts:.1f}s)')
                pd.DataFrame(wf_rows).to_csv(f'{OUT_DIR}/_lin_wf_rows_partial.csv', index=False)
    print(f'\nWalk-forward sweep done: {len(wf_rows)} rows '
          f'({len(wf_rows)//2} fits) in {(time.time()-_t0)/60:.1f} min')
    if _wf_failed:
        print(f'  {len(_wf_failed)} cells failed: {_wf_failed}')

df_wf = pd.DataFrame(wf_rows)
print(f'df_wf shape: {df_wf.shape}')

### Block-CV / leave-one-regime-out runner

In [ ]:
# ── block-CV / leave-one-regime-out runner (NON-CAUSAL diagnostic) ──────
# IDENTICAL fold geometry and two-sided purge+embargo to v5's run_security_blockcv —
# trains on data SURROUNDING the held-out block (including the FUTURE), so it is a
# learnability / regime-transfer diagnostic, NOT a forecast (see the title cell's
# asymmetric interpretation rule). The only change is the estimator (fit_linear) and the
# threaded model_kind. gap = (H−1) + max(H,10) business days around EVERY test segment.

def _blockcv_entries(scheme):
    """Return [(label, regime, [(start_ts, end_ts), ...]), ...] for the scheme."""
    if scheme == 'block_cv':
        return [(name, regime, [(pd.Timestamp(s), pd.Timestamp(e))])
                for (name, s, e, regime) in BLOCK_CV_FOLDS]
    if scheme == 'loro':
        out = []
        for regime in dict.fromkeys(r for (_n, _s, _e, r) in BLOCK_CV_FOLDS):
            segs = [(pd.Timestamp(s), pd.Timestamp(e))
                    for (_n, s, e, r) in BLOCK_CV_FOLDS if r == regime]
            out.append((f'LORO_{regime}', regime, segs))
        return out
    raise ValueError(scheme)


def run_security_blockcv(sec, df_raw, scheme, H, model_kind, verbose=False):
    """Block-CV / LORO across blocks for one security at horizon H, for ONE estimator.
    Two-sided purge+embargo around every test segment. Emits train+oos rows per block."""
    if sec not in df_raw.columns:
        return [], []
    panel = build_panel_v4(df_raw, [sec], H)
    if len(panel) == 0:
        return [], []
    fc  = [c for c in panel.columns if c not in META_COLS]
    gap = (H - 1) + EMBARGO_FOR_H(H)                       # business days, scales with H
    dates = panel['date']
    rows, fits = [], []
    for label, regime, segs in _blockcv_entries(scheme):
        test_mask = pd.Series(False, index=panel.index)
        for (s, e) in segs:
            test_mask |= (dates >= s) & (dates <= e)
        train_mask = ~test_mask
        for (s, e) in segs:                                # symmetric gap, both edges
            lo = s - pd.tseries.offsets.BDay(gap)
            hi = e + pd.tseries.offsets.BDay(gap)
            train_mask &= ~((dates >= lo) & (dates <= hi))
        tr = panel[train_mask].copy()
        te = panel[test_mask].copy()
        if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
            if verbose:
                print(f'  {scheme}/{label}: skip (train={len(tr)}, test={len(te)})')
            continue
        try:
            r, fi = _fit_score_rows(tr, te, fc, H, model_kind, sec, scheme, label, regime)
        except Exception as ex:
            if verbose:
                print(f'  {scheme}/{label}: FAILED {repr(ex)[:70]}')
            continue
        rows.extend(r); fits.append(fi)
        if verbose:
            oos = r[1]
            print(f'  {scheme}/{label:<12s} train DA={r[0]["dir_acc"]:.3f}  '
                  f'OOS DA={oos["dir_acc"]:.3f}  n_train={len(tr)}  n_test={oos["n_obs"]}')
    return rows, fits


print('Block-CV/LORO linear runner defined: run_security_blockcv (two-sided purge+embargo,')
print('  per estimator; non-causal learnability diagnostic — NOT a forecast)')

### Block-CV / LORO sweep

In [ ]:
# ── BLOCK-CV / LORO SWEEP — model_kind × schemes × H × tenors × blocks ──
# Same fold geometry as v5's block sweep, looped per estimator. Feeds a SEPARATE MTC
# family (learnability), never Part C, never the headline. Gated by RUN_BLOCK_CV / RUN_LORO.
block_rows, block_fits = [], []
_block_failed = []
_schemes = ([] + (['block_cv'] if RUN_BLOCK_CV else []) + (['loro'] if RUN_LORO else []))

if not _schemes:
    print('[skipped] RUN_BLOCK_CV = RUN_LORO = False  '
          '(set RUN_FULL_SWEEP=True to run the non-causal diagnostics)')
else:
    _t0 = time.time()
    for mk in MODEL_KINDS:
        for scheme in _schemes:
            for H_run in HORIZONS:
                for sec in SWEEP_TENORS:
                    _ts = time.time()
                    try:
                        rows, fits = run_security_blockcv(sec, df_raw, scheme, H_run, mk)
                    except Exception as e:
                        print(f'  {mk} {scheme} H={H_run} {sec}: FAILED {repr(e)[:60]}')
                        _block_failed.append((mk, scheme, H_run, sec)); continue
                    block_rows.extend(rows); block_fits.extend(fits)
                    _oos = [r for r in rows if r['sample'] == 'oos']
                    _da = ', '.join(f'{r["dir_acc"]:.3f}' for r in _oos)
                    print(f'  {mk:<10s} {scheme:<9s} H={H_run:<3d} {sec:<10s} '
                          f'blocks={len(_oos)} OOS_DA=[{_da}]  ({time.time()-_ts:.1f}s)')
                    pd.DataFrame(block_rows).to_csv(f'{OUT_DIR}/_lin_block_rows_partial.csv', index=False)
    print(f'\nBlock-CV/LORO sweep done: {len(block_rows)} rows in {(time.time()-_t0)/60:.1f} min')
    if _block_failed:
        print(f'  {len(_block_failed)} cells failed: {_block_failed}')

df_block = pd.DataFrame(block_rows)
print(f'df_block shape: {df_block.shape}')

### Assemble results and apply multiple-testing correction

In [ ]:
# ── assemble lin_metrics_long.csv + apply MTC in SEPARATE families ──
# Walk-forward (forecastability) and block-CV/LORO (learnability) are corrected in their
# OWN families — exactly v5's family split — but here within EACH estimator, so the
# OLS and ElasticNet family sizes are each comparable to HTBoost's single-estimator
# family. The SHARED_COLS schema is preserved so the linear master file concatenates
# with v5_metrics_long.csv with zero reconciliation.
_parts = [d for d in (df_wf, df_block) if len(d) > 0]
df_all = pd.concat(_parts, ignore_index=True) if _parts else pd.DataFrame(columns=SHARED_COLS)

for c in SHARED_COLS:
    if c not in df_all.columns:
        df_all[c] = np.nan
for c in ('pca_k', 'alpha', 'l1_ratio', 'n_nonzero'):
    if c not in df_all.columns:
        df_all[c] = np.nan
for c in ('reject_bonferroni', 'reject_fdr_bh', 'mtc_N', 'mtc_family'):
    if c not in df_all.columns:
        df_all[c] = (False if 'reject' in c else (np.nan if c == 'mtc_N' else ''))


# apply_mtc_family is the SHARED reconciled harness function; linear passes
# model_kind=mk so each estimator gets its own per-estimator MTC family.
from src.evaluation.metrics import apply_mtc_family


mtc_summary = []
for mk in MODEL_KINDS:
    N_wf, b_wf, h_wf = apply_mtc_family(
        df_all, ['walk_forward'], f'walk_forward[{mk}]:{{horizon×tenor×regime}}', model_kind=mk)
    N_bl, b_bl, h_bl = apply_mtc_family(
        df_all, ['block_cv', 'loro'], f'noncausal[{mk}]:{{scheme×horizon×tenor×block}}', model_kind=mk)
    mtc_summary.append((mk, 'walk_forward', N_wf, b_wf, h_wf))
    mtc_summary.append((mk, 'noncausal',    N_bl, b_bl, h_bl))

_extra = [c for c in df_all.columns if c not in SHARED_COLS]
df_all = df_all[[c for c in SHARED_COLS if c in df_all.columns] + _extra]
df_all.to_csv(f'{OUT_DIR}/lin_metrics_long.csv', index=False)

print('=== lin_metrics_long.csv written ===')
print(f'  rows={len(df_all)}  cols={df_all.shape[1]}  (shared={len(SHARED_COLS)})')
for mk, fam, N, b, h in mtc_summary:
    print(f'  MTC {mk:<10s} {fam:<13s}: N={N:<4d} Bonferroni survivors={b}  BH-FDR={h}')
print(f'  Saved: {OUT_DIR}/lin_metrics_long.csv')
if len(df_all) == 0:
    print('  [note] sweeps were gated OFF (RUN_FULL_SWEEP=False) → master file is the '
          'empty SHARED schema. Flip RUN_FULL_SWEEP=True to populate it.')

### Report — overfit gap, aggregates, figures 1–3

In [ ]:
# ── report 1 — master overfit-gap table, aggregates, figures 1–3 ─────────────────────
# Ch.7-vs-Ch.8 headline for the linear benchmarks: train and OOS DirAcc/R² side by side
# with an explicit overfit-gap column, PER estimator. Walk-forward only (the causal
# finding). All CSVs carry a model_kind column so they slot beside the v5 aggregates.

# save_fig now lives in thesis_style (generalised to take fig_dir explicitly).
from thesis_style import save_fig


_OVERFIT_COLS = ['model_kind', 'security', 'tenor', 'horizon', 'fold', 'regime',
                 'dir_acc_train', 'dir_acc_oos', 'gap_dir_acc',
                 'r2_raw_train', 'r2_raw_oos', 'gap_r2',
                 'ct_r2_oos_train', 'ct_r2_oos_oos', 'n_obs_train', 'n_obs_oos']


def _pair_train_oos(df, schemes):
    d = df[df['validation_scheme'].isin(schemes)].copy()
    if len(d) == 0:
        return pd.DataFrame(columns=_OVERFIT_COLS)         # header-only, never zero-column
    keys = ['model_kind', 'security', 'tenor', 'horizon', 'fold', 'regime']
    piv = d.pivot_table(index=keys, columns='sample',
                        values=['dir_acc', 'r2_raw', 'ct_r2_oos', 'n_obs'], aggfunc='first')
    piv.columns = [f'{a}_{b}' for a, b in piv.columns]
    piv = piv.reset_index()
    if 'dir_acc_train' in piv and 'dir_acc_oos' in piv:
        piv['gap_dir_acc'] = piv['dir_acc_train'] - piv['dir_acc_oos']
    if 'r2_raw_train' in piv and 'r2_raw_oos' in piv:
        piv['gap_r2'] = piv['r2_raw_train'] - piv['r2_raw_oos']
    return piv


master_wf = _pair_train_oos(df_all, ['walk_forward'])
master_wf.to_csv(f'{OUT_DIR}/lin_master_overfit_gap.csv', index=False)
print('=== master overfit-gap table (walk-forward) ===')
if len(master_wf):
    _show = [c for c in ['model_kind', 'security', 'horizon', 'fold', 'regime',
                         'dir_acc_train', 'dir_acc_oos', 'gap_dir_acc',
                         'ct_r2_oos_oos', 'n_obs_oos'] if c in master_wf.columns]
    print(master_wf[_show].head(40).to_string(index=False, float_format=lambda x: f'{x:.3f}'))
else:
    print('  (no walk-forward rows — sweep gated OFF; file written with header only)')
print(f'  Saved: {OUT_DIR}/lin_master_overfit_gap.csv')

# Aggregates by horizon and by regime (walk-forward OOS), per estimator.
oos_wf = df_all[(df_all['validation_scheme'] == 'walk_forward') & (df_all['sample'] == 'oos')]
agg_h = pd.DataFrame()
if len(oos_wf):
    agg_h = oos_wf.groupby(['model_kind', 'horizon']).agg(
        n=('dir_acc', 'size'), mean_DA=('dir_acc', 'mean'), std_DA=('dir_acc', 'std'),
        pct_gt50=('dir_acc', lambda s: float((s > 0.5).mean() * 100)),
        mean_CT_R2=('ct_r2_oos', 'mean'),
        bonf=('reject_bonferroni', 'sum'), bh=('reject_fdr_bh', 'sum')).reset_index()
    agg_h.to_csv(f'{OUT_DIR}/lin_agg_by_horizon.csv', index=False)
    print('\n=== Aggregate by HORIZON (walk-forward OOS) ===')
    print(agg_h.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

    agg_r = oos_wf.groupby(['model_kind', 'regime']).agg(
        n=('dir_acc', 'size'), mean_DA=('dir_acc', 'mean'), std_DA=('dir_acc', 'std'),
        pct_gt50=('dir_acc', lambda s: float((s > 0.5).mean() * 100)),
        mean_CT_R2=('ct_r2_oos', 'mean')).reset_index()
    agg_r.to_csv(f'{OUT_DIR}/lin_agg_by_regime.csv', index=False)
    print('\n=== Aggregate by REGIME (walk-forward OOS) ===')
    print(agg_r.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
else:
    pd.DataFrame(columns=['model_kind', 'horizon', 'n', 'mean_DA', 'std_DA',
                          'pct_gt50', 'mean_CT_R2', 'bonf', 'bh']
                 ).to_csv(f'{OUT_DIR}/lin_agg_by_horizon.csv', index=False)
    pd.DataFrame(columns=['model_kind', 'regime', 'n', 'mean_DA', 'std_DA',
                          'pct_gt50', 'mean_CT_R2']
                 ).to_csv(f'{OUT_DIR}/lin_agg_by_regime.csv', index=False)
    print('\n  (walk-forward OOS empty — aggregate CSVs written with headers only)')

# ── Figure 1 — train vs OOS DirAcc distributions per estimator ───────────────────────
if len(master_wf) and {'dir_acc_train', 'dir_acc_oos'} <= set(master_wf.columns):
    fig, axes = plt.subplots(1, len(MODEL_KINDS), figsize=(5.2 * len(MODEL_KINDS), 4.2),
                             squeeze=False)
    for j, mk in enumerate(MODEL_KINDS):
        ax = axes[0, j]
        d = master_wf[master_wf['model_kind'] == mk]
        ax.hist(d['dir_acc_train'].dropna(), bins=12, alpha=0.6, color=CB['OLS'] if mk == 'OLS' else CB['ElasticNet'], label='train')
        ax.hist(d['dir_acc_oos'].dropna(), bins=12, alpha=0.55, color='#444444', label='OOS')
        ax.axvline(0.5, color='red', ls='--', lw=1, label='chance')
        ax.set_title(f'{mk}: train vs OOS DirAcc'); ax.set_xlabel('directional accuracy')
        ax.legend()
    fig.suptitle('Linear benchmarks — in-sample vs out-of-sample directional accuracy', y=1.02)
    save_fig(fig, 'fig_lin_train_oos_diracc',
             'Distribution of in-sample (train) versus out-of-sample (OOS) directional '
             'accuracy across walk-forward folds, horizons and Norwegian tenors, for each '
             'linear benchmark. The dashed red line marks the 0.50 chance level. A train '
             'distribution shifted right of the OOS distribution is the overfit gap; OOS '
             'mass centred on 0.50 is the null-consistent benchmark result.', FIG_DIR)

# ── Figure 2 — OOS DirAcc vs horizon (±1 SD) per estimator ───────────────────────────
if len(oos_wf):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for mk in MODEL_KINDS:
        d = oos_wf[oos_wf['model_kind'] == mk]
        m = d.groupby('horizon')['dir_acc'].mean()
        s = d.groupby('horizon')['dir_acc'].std()
        col = CB['OLS'] if mk == 'OLS' else CB['ElasticNet']
        ax.errorbar(m.index, m.values, yerr=s.values, fmt='o-', capsize=4, color=col, label=mk)
    ax.axhline(0.5, color=CB['chance'], ls='--', lw=1, label='chance (0.50)')
    ax.set_xscale('log'); ax.set_xticks(sorted(oos_wf['horizon'].unique()))
    ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel('forecast horizon (business days)'); ax.set_ylabel('OOS directional accuracy')
    ax.set_title('OOS directional accuracy vs horizon — linear benchmarks'); ax.legend()
    save_fig(fig, 'fig_lin_oos_diracc_by_horizon',
             'Out-of-sample directional accuracy versus forecast horizon (log scale) for '
             'OLS and Elastic Net, averaged over the Norwegian tenors and walk-forward '
             'folds, with ±1 standard-deviation whiskers. The dashed line is the 0.50 '
             'chance level. Designed for direct overlay against the HTBoost panel.', FIG_DIR)

# ── Figure 3 — mean Campbell–Thompson OOS R² vs horizon per estimator ────────────────
if len(oos_wf):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for mk in MODEL_KINDS:
        d = oos_wf[oos_wf['model_kind'] == mk]
        m = d.groupby('horizon')['ct_r2_oos'].mean()
        col = CB['OLS'] if mk == 'OLS' else CB['ElasticNet']
        ax.plot(m.index, m.values, 'o-', color=col, label=mk)
    ax.axhline(0.0, color=CB['zero'], ls='--', lw=1, label='zero (= random walk)')
    ax.set_xscale('log'); ax.set_xticks(sorted(oos_wf['horizon'].unique()))
    ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel('forecast horizon (business days)'); ax.set_ylabel('mean Campbell–Thompson OOS R²')
    ax.set_title('Campbell–Thompson OOS R² vs horizon — linear benchmarks'); ax.legend()
    save_fig(fig, 'fig_lin_ct_r2_by_horizon',
             'Mean Campbell–Thompson out-of-sample R² (1 − MSE_model / MSE_randomwalk) '
             'versus forecast horizon for OLS and Elastic Net. Values at or below zero '
             'indicate the linear model does not beat the no-change random-walk benchmark.', FIG_DIR)
print('\n[REPORT 1] master overfit-gap + aggregates + figures 1–3 done.')

### Report — coefficients, selected penalties, figures 4–5, manifest

In [ ]:
# ── report 2 — coefficient buckets, selected penalties, block-CV vs WF, figs 4–5, manifest
# Linear "feature relevance": standardised coefficient magnitude per taxonomy bucket
# (comparable to HTBoost's relevance buckets in Ch.7), plus Elastic Net sparsity.
# Falls back to the smoke fits when the full sweep is gated OFF, so the report and its
# figures are always populated; the source is recorded in the manifest.
report_fits = list(wf_fits)
if not report_fits and 'smoke_fits' in dir():
    for _mk, (_rows, _fits) in smoke_fits.items():
        report_fits.extend(_fits)
    _fits_source = 'smoke (one tenor/horizon, all folds) — full sweep gated OFF'
else:
    _fits_source = 'walk-forward sweep'
print(f'Coefficient/penalty reports built from: {_fits_source}  ({len(report_fits)} fits)')

# ── lin_coef_summary.csv — |coef| mass by taxonomy bucket, per (model_kind, tenor, horizon)
_BUCKETS = ['curve', 'momentum', 'vol', 'macro', 'credit', 'cross_market', 'norway', 'carry_roll']
_coef_recs = []
for f in report_fits:
    abs_by_b = {b: 0.0 for b in _BUCKETS}
    nz_by_b  = {b: 0 for b in _BUCKETS}
    for feat, val in f['coef'].items():
        b = bucket_feature(feat)
        abs_by_b[b] = abs_by_b.get(b, 0.0) + abs(float(val))
        nz_by_b[b]  = nz_by_b.get(b, 0) + int(abs(float(val)) > 1e-12)
    for b in _BUCKETS:
        _coef_recs.append({'model_kind': f['model_kind'], 'tenor': f['tenor'],
                           'horizon': f['horizon'], 'bucket': b,
                           'abs_coef': abs_by_b[b], 'n_nonzero': nz_by_b[b]})
coef_df = pd.DataFrame(_coef_recs)
if len(coef_df):
    coef_summary = coef_df.groupby(['model_kind', 'tenor', 'horizon', 'bucket']).agg(
        mean_abs_coef=('abs_coef', 'mean'), mean_nonzero=('n_nonzero', 'mean'),
        n_fits=('abs_coef', 'size')).reset_index()
else:
    coef_summary = pd.DataFrame(columns=['model_kind', 'tenor', 'horizon', 'bucket',
                                         'mean_abs_coef', 'mean_nonzero', 'n_fits'])
coef_summary.to_csv(f'{OUT_DIR}/lin_coef_summary.csv', index=False)
print(f'  Saved: {OUT_DIR}/lin_coef_summary.csv  ({len(coef_summary)} rows)')

# ── lin_elasticnet_selected_params.csv — chosen alpha/l1_ratio per fold (appendix) ──
_en_recs = [{'model_kind': f['model_kind'], 'tenor': f['tenor'], 'horizon': f['horizon'],
             'fold': f['fold'], 'regime': f['regime'], 'scheme': f['scheme'],
             'alpha': f['alpha'], 'l1_ratio': f['l1_ratio'], 'n_nonzero': f['n_nonzero'],
             'feature_count': f['feature_count']}
            for f in report_fits if f['model_kind'] == 'ElasticNet']
en_params = pd.DataFrame(_en_recs)
en_params.to_csv(f'{OUT_DIR}/lin_elasticnet_selected_params.csv', index=False)
print(f'  Saved: {OUT_DIR}/lin_elasticnet_selected_params.csv  ({len(en_params)} rows)')
if len(en_params):
    print(en_params.head(12).to_string(index=False, float_format=lambda x: f'{x:.4g}'))

# ── lin_blockcv_vs_walkforward.csv — learnability vs forecastability, per estimator ──
def _scheme_oos_mean(df, mk, scheme):
    d = df[(df['sample'] == 'oos') & (df['model_kind'] == mk)
           & (df['validation_scheme'] == scheme)]
    return d.groupby(['tenor', 'regime'])['dir_acc'].mean() if len(d) else pd.Series(dtype=float)


_cmp_parts = []
for mk in MODEL_KINDS:
    wf_m = _scheme_oos_mean(df_all, mk, 'walk_forward')
    if not len(wf_m):
        continue
    cmp_mk = pd.DataFrame({'walk_forward_OOS': wf_m})
    bl_m = _scheme_oos_mean(df_all, mk, 'block_cv')
    lo_m = _scheme_oos_mean(df_all, mk, 'loro')
    if len(bl_m):
        cmp_mk['block_cv_OOS'] = bl_m
        cmp_mk['gap_block_minus_wf'] = cmp_mk['block_cv_OOS'] - cmp_mk['walk_forward_OOS']
    if len(lo_m):
        cmp_mk['loro_OOS'] = lo_m
    cmp_mk.insert(0, 'model_kind', mk)
    _cmp_parts.append(cmp_mk.reset_index())
cmp_block = pd.concat(_cmp_parts, ignore_index=True) if _cmp_parts else \
    pd.DataFrame(columns=['model_kind', 'tenor', 'regime', 'walk_forward_OOS'])
cmp_block.to_csv(f'{OUT_DIR}/lin_blockcv_vs_walkforward.csv', index=False)
print(f'  Saved: {OUT_DIR}/lin_blockcv_vs_walkforward.csv  ({len(cmp_block)} rows)')

# ── Figure 4 — OLS / ElasticNet / HTBoost OOS DirAcc by horizon (key Ch.8 figure) ──
_fig4_status = 'skipped'
oos_wf2 = df_all[(df_all['validation_scheme'] == 'walk_forward') & (df_all['sample'] == 'oos')]
if os.path.exists(HTB_METRICS_CSV) and len(oos_wf2):
    htb = pd.read_csv(HTB_METRICS_CSV)
    htb_oos = htb[(htb.get('validation_scheme') == 'walk_forward') & (htb['sample'] == 'oos')]
    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    for mk in MODEL_KINDS:
        d = oos_wf2[oos_wf2['model_kind'] == mk]
        m = d.groupby('horizon')['dir_acc'].mean()
        ax.plot(m.index, m.values, 'o-', color=CB['OLS'] if mk == 'OLS' else CB['ElasticNet'], label=mk)
    if len(htb_oos):
        mh = htb_oos.groupby('horizon')['dir_acc'].mean()
        ax.plot(mh.index, mh.values, 's--', color=CB['HTBoost'], label='HTBoost')
    ax.axhline(0.5, color=CB['chance'], ls=':', lw=1, label='chance (0.50)')
    ax.set_xscale('log'); ax.set_xticks(sorted(oos_wf2['horizon'].unique()))
    ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel('forecast horizon (business days)'); ax.set_ylabel('OOS directional accuracy')
    ax.set_title('OOS directional accuracy by horizon — linear vs HTBoost'); ax.legend()
    save_fig(fig, 'fig_lin_vs_htb_diracc',
             'Out-of-sample directional accuracy by forecast horizon for OLS, Elastic Net '
             'and HTBoost, averaged over the Norwegian tenors and walk-forward folds. The '
             'dotted line marks 0.50. This is the key Chapter 8 comparison of the '
             'nonlinear model against its linear benchmarks on identical data and folds.')
    _fig4_status = 'written'
else:
    print(f'  [fig 4] skipped — '
          f'{"HTBoost CSV absent" if not os.path.exists(HTB_METRICS_CSV) else "no linear walk-forward rows"}')

# ── Figure 5 — Elastic Net coefficient mass by taxonomy bucket, per horizon ──────────
_fig5_status = 'skipped'
en_coef = coef_summary[coef_summary['model_kind'] == 'ElasticNet']
if len(en_coef):
    piv = en_coef.groupby(['horizon', 'bucket'])['mean_abs_coef'].mean().unstack('bucket').fillna(0.0)
    piv = piv.reindex(columns=[b for b in _BUCKETS if b in piv.columns])
    share = piv.div(piv.sum(axis=1).replace(0, np.nan), axis=0) * 100
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    bottom = np.zeros(len(share))
    for k, b in enumerate(share.columns):
        ax.bar([str(h) for h in share.index], share[b].values, bottom=bottom,
               color=OKABE_ITO[k % len(OKABE_ITO)], label=b)
        bottom += share[b].values
    ax.set_xlabel('forecast horizon (business days)')
    ax.set_ylabel('share of |coefficient| mass (%)')
    ax.set_title('Elastic Net — coefficient mass by taxonomy bucket')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax.grid(axis='x', visible=False)
    save_fig(fig, 'fig_lin_coef_buckets',
             'Composition of Elastic Net standardised-coefficient magnitude by feature '
             'taxonomy bucket, as a share of total |coefficient| mass, for each forecast '
             'horizon (averaged over Norwegian tenors and folds). Indicates where the '
             'penalised linear model places weight — the linear analogue of HTBoost '
             'feature relevance.')
    _fig5_status = 'written'
else:
    print('  [fig 5] skipped — no Elastic Net coefficients available')

# ── MANIFEST ─────────────────────────────────────────────────────────────────────────
_lin_extra = [c for c in df_all.columns if c not in SHARED_COLS]
_missing_from_lin = [c for c in SHARED_COLS if c not in df_all.columns]
_manifest = []
_manifest.append('# Linear benchmarks (Elastic Net + OLS) — output manifest\n')
_manifest.append(f'_Generated {RUN_TS} · notebook={NOTEBOOK_TAG} · estimators={MODEL_KINDS}_\n')
_manifest.append(f'Coefficient/penalty reports source: **{_fits_source}**.  '
                 f'Full sweep gated by `RUN_FULL_SWEEP` (currently '
                 f'**{"ON" if RUN_FULL_SWEEP else "OFF"}**).\n')
_manifest.append('\n## CSV outputs → Chapter mapping\n')
_csv_map = [
    ('lin_metrics_long.csv', 'master long table (model_kind × security × horizon × fold × sample)', 'Ch.7 §results, Ch.8 §comparison'),
    ('lin_master_overfit_gap.csv', 'paired train-vs-OOS DirAcc/R² with gap columns', 'Ch.7 §overfitting / Ch.8 §generalisation'),
    ('lin_agg_by_horizon.csv', 'OOS aggregates per estimator × horizon (mean_DA, pct_gt50, CT-R², Bonf/BH)', 'Ch.8 §horizon results'),
    ('lin_agg_by_regime.csv', 'OOS aggregates per estimator × regime', 'Ch.8 §regime robustness'),
    ('lin_blockcv_vs_walkforward.csv', 'learnability (block-CV/LORO) vs forecastability (WF)', 'Ch.8 §causal-vs-noncausal'),
    ('lin_coef_summary.csv', 'standardised |coef| by 8 taxonomy buckets (+ EN sparsity)', 'Ch.7 §feature relevance'),
    ('lin_elasticnet_selected_params.csv', 'chosen alpha/l1_ratio per fold', 'Appendix (model selection)'),
]
def _nrows(path):
    """Row count, robust to header-only / empty CSVs."""
    if not os.path.exists(path):
        return 'MISSING'
    try:
        return len(pd.read_csv(path))
    except pd.errors.EmptyDataError:
        return 0


for fn, desc, ch in _csv_map:
    _manifest.append(f'- `{fn}` — {desc}  → _{ch}_  (rows={_nrows(f"{OUT_DIR}/{fn}")})\n')
_manifest.append('\n## Figures (vector PDF + caption .txt) → Chapter mapping\n')
_fig_map = [
    ('fig_lin_train_oos_diracc.pdf', 'train vs OOS DirAcc distributions per estimator', 'Ch.7 §overfitting'),
    ('fig_lin_oos_diracc_by_horizon.pdf', 'OOS DirAcc vs horizon (±1 SD), OLS & EN', 'Ch.8 §horizon results'),
    ('fig_lin_ct_r2_by_horizon.pdf', 'mean CT OOS R² vs horizon per estimator', 'Ch.8 §forecast skill'),
    ('fig_lin_vs_htb_diracc.pdf', f'OLS/EN/HTBoost OOS DirAcc overlay ({_fig4_status})', 'Ch.8 §key comparison'),
    ('fig_lin_coef_buckets.pdf', f'EN coefficient mass by bucket ({_fig5_status})', 'Ch.7 §feature relevance'),
]
for fn, desc, ch in _fig_map:
    _p = f'{FIG_DIR}/{fn}'
    _manifest.append(f'- `figures/{fn}` — {desc}  → _{ch}_  '
                     f'({"present" if os.path.exists(_p) else "absent"})\n')
_manifest.append('\n## Schema check vs SHARED_COLS (pooled-comparability contract)\n')
_manifest.append(f'- SHARED_COLS columns present in lin_metrics_long: '
                 f'{len(SHARED_COLS) - len(_missing_from_lin)}/{len(SHARED_COLS)}\n')
_manifest.append(f'- SHARED_COLS columns MISSING from lin table: '
                 f'{_missing_from_lin if _missing_from_lin else "none — full match"}\n')
_manifest.append(f'- Extra linear-only columns (beyond SHARED_COLS): {_lin_extra}\n')
_manifest.append('  (extras are appendix side-columns: pca_k, alpha, l1_ratio, n_nonzero, '
                 'and the MTC reject_* / mtc_* columns — also present in v5.)\n')
with open(f'{OUT_DIR}/lin_manifest.md', 'w') as fh:
    fh.write('\n'.join(_manifest))

print('\n' + '=' * 78)
print('MANIFEST'); print('=' * 78)
print('\n'.join(_manifest))
print(f'[REPORT 2] coef summary + selected params + block-CV table + figs 4–5 + manifest done.')
print(f'  Schema match vs SHARED_COLS: '
      f'{"FULL" if not _missing_from_lin else "MISSING " + str(_missing_from_lin)}')
print(f'  Saved manifest: {OUT_DIR}/lin_manifest.md')